# PINN inversion of a spatially variable bottom friction — twin experiment (realistic-noise edition)

**Question.** With the *same* four-parameter zonal parameterisation, the same data windows, the same noise realisations (common random numbers) and the same identification rule as the classical arm of the twin notebook, does a physics-informed neural network (Adam only) recover the friction structure from sea-level data — and does it do so as often as the classical arm? If it fails where the classical arm succeeds, the failure is an optimisation problem of the PINN, not an information limit.

**What is reused, verbatim, from `hormuz-twin-experiment-realistic-noise-cl10-0802.ipynb`:** solver and problem definition, the absolute and space-time-correlated noise machinery (levels L0-L3, exact GLS whitening), the data windows, the independent fine-grid RK4 generator, the classical multi-start rows, the solver/noise gate and the generator gate. The PINN is the only new component.

**STATUS OF THIS FILE.** Written without being able to run JAX where it was authored. Tested in NumPy: the continuous residual operator on the generator solution (G2), the hard boundary/initial constraints, the continuous friction field against `cf_zonal`, the data builder and the whitened misfit. **Untested until you run stage G: everything that uses JAX** (network, forward-mode derivatives, Adam loop, schedules). Treat any error in stage G as expected debugging, not as a result.

**Design decisions (agreed before writing).**
1. *Primary level L1, U0 = 1.* Classical h-only identifies 10/10 there and 0-2/10 at L2, so only L1 gives a non-trivial comparison. L2 is a secondary stage.
2. *Same four parameters* `(ln Cf_low, ln Cf_high, b0, s)`, early window, 3 h ramp (hard initial conditions need a ramp; cold start is not supported), Adam only (Radfar 2026), float32.
3. *Thresholds are fixed in the pre-registration hash before any inverse row.* Hyper-parameters may be tuned **only on the forward test** (theta known, generator data as truth); they are hashed, and inverse rows refuse to run unless the gate was passed under the same hash. Every gate attempt is logged.
4. *PINN data arm = independent generator* (continuous-equation truth, no inverse crime). The classical reference (10/10) uses the coarse solver (inverse crime, best case); the classical control of stage K uses the generator as well, so all three numbers are on the table.

**Stages** (`STAGE` in cell 1): `G` gates (G1 autodiff, G2 residual operator, G3 forward test, G4 positive control) | `R` PINN rows at L1 (h and h+u+v) | `S` PINN rows at L2 | `K` classical control rows with independent data | `ALL` (quick only).

**Gates (all must pass; a failed gate blocks the rows).** G1 autodiff derivative = float64 finite difference. G2 PDE residual operator annihilates the generator solution (equation, sign and scale check; also sensitive to friction). G3 forward test with known friction, PDE only: rms error of h <= 0.5 sd of the SLA noise, of u <= 0.5 sd of the velocity noise, relative error of v <= 0.5 (no momentum collapse). G4 positive control: starting from the forward solution and the true parameters, theta must stay identified with data of realisation 0.

**Honest warning on G3 at L1.** 0.5 sd of the L1 sea-level noise is 1 cm on a 2.8 m signal (0.36%). That is a demanding accuracy for a PINN; failing it is a legitimate outcome, and the achieved error is printed in units of every level's sd so that the next decision (more steps, larger network, or level L2) is made on numbers. **ASSUMPTION WARNING:** noise levels are order-of-magnitude assumptions, not measurements of this project.


In [1]:
import os, json, time, hashlib, math, glob
import numpy as np
from scipy.optimize import minimize
from scipy import stats
try:
    import jax, jax.numpy as jnp
    from jax import lax
    jax.config.update("jax_enable_x64", True)   # needed by the classical solver (control rows); the PINN itself runs in float32 with explicit casts
    HAVE_JAX = True
except Exception as _e:
    HAVE_JAX = False
    print("WARNING: JAX not available ->", _e, "\n  PINN training, the classical control rows and the JAX part of the gate are skipped; NumPy-only checks still run.")

# ------------------------------------------------------------------------------------------------------------------
# The only things you normally change between sessions: MODE, STAGE (and, for the secondary rows, nothing else).
# ------------------------------------------------------------------------------------------------------------------
MODE = "standard"             # "quick" (pipeline smoke test, ~15-25 min) | "standard"
STAGE = "G"              # "G" gates | "R" PINN rows, primary (level L1) | "S" PINN rows, secondary (level L2) | "K" classical control rows (independent data) | "ALL" (quick only)
NOISE_LEVEL = "L1"         # primary noise level of this notebook (agreed: classical h-only identifies at L0/L1 but not at L2)
WEIGHTING = "gls"          # "gls": the inversion knows the noise covariance | "white": it ignores the correlation
ARM_PINN = "indep"         # data of the PINN rows: "indep" = independent fine-grid RK4 generator (continuous-equation truth, no inverse crime) | "same" = coarse solver of the inversion
RUN = dict(GATE=STAGE in ("G", "K", "ALL"), E6=STAGE in ("G", "K", "ALL"))   # solver/noise gate and generator gate of the twin notebook (reused verbatim below)
RESUME = True
RAMP_DEFAULT = 3.0         # the PINN uses hard initial conditions and therefore needs the 3 h ramp (cold start is NOT supported here)
RAMP_HOURS = RAMP_DEFAULT

# ------------------------------------------------------------------------------------------------------------------
# OBSERVATION-NOISE MODEL: identical to the realistic-noise twin notebook (copied, not edited). ASSUMPTION WARNING: order-of-magnitude assumptions.
# ------------------------------------------------------------------------------------------------------------------
NOISE_LEVELS = {
    "L0": dict(sig_h_white=0.01, sig_h_corr=0.00, sig_vel=0.005, desc="diagnostic floor: 1 cm white SLA, 0.5 cm/s velocity (asks whether ANY noise level makes h-only identifiable)"),
    "L1": dict(sig_h_white=0.02, sig_h_corr=0.00, sig_vel=0.010, desc="optimistic: 2 cm white SLA, 1 cm/s velocity"),
    "L2": dict(sig_h_white=0.05, sig_h_corr=0.03, sig_vel=0.030, desc="baseline of the twin notebook: 5 cm white + 3 cm correlated SLA (25 km, 3 h), 3 cm/s velocity"),
    "L3": dict(sig_h_white=0.10, sig_h_corr=0.06, sig_vel=0.060, desc="pessimistic: 10 cm white + 6 cm correlated SLA, 6 cm/s velocity"),
}
NOISE_CORR = dict(L_space_km=25.0, tau_time_h=3.0)
assert NOISE_LEVEL in NOISE_LEVELS and WEIGHTING in ("gls", "white") and ARM_PINN in ("indep", "same")
NOISE = dict(level=NOISE_LEVEL)
def set_noise(level):
    assert level in NOISE_LEVELS
    NOISE["level"] = level

# ------------------------------------------------------------------------------------------------------------------
# PINN HYPER-PARAMETERS. RULE: these may be tuned ONLY on the forward test (stage G, theta known, generator data as truth). They are frozen
# (hashed) before any inverse row is run; the inverse rows refuse to run unless the gate has been passed under the SAME hash.
# ------------------------------------------------------------------------------------------------------------------
PINN_STD = dict(width=128, depth=5, n_harm=3, steps=40000, block=500, n_col=8192,
                lr0=1e-3, lr1=2e-5, warm=500, lam_p0=1.0, lam_p1=1000.0, f_lam=0.6, f_freeze=0.25, f_anneal=0.70,
                lr_theta=3e-3, w_start=0.30, lam_v=1.0, v_scale="natural", hs_factor=3.0, us_factor=1.0, vs_factor=0.01, t_g_h=1.0, net_seed=0,
                ctrl_steps_frac=0.5, ctrl_lr0=2e-4)
PINN_QUICK = dict(PINN_STD, width=48, depth=3, steps=1000, block=100, n_col=1024, warm=100, ctrl_steps_frac=1.0)
PINN = dict(PINN_QUICK if MODE == "quick" else PINN_STD)
assert PINN["steps"] % PINN["block"] == 0
PINN_HASH = hashlib.sha256(json.dumps(dict(PINN, mode=MODE, level=NOISE_LEVEL, arm=ARM_PINN, weighting=WEIGHTING), sort_keys=True).encode()).hexdigest()[:16]

_NOISE_PRE = dict(levels=NOISE_LEVELS, corr=NOISE_CORR, level=NOISE_LEVEL, weighting=WEIGHTING,
                  text="absolute noise, identical to the twin notebook; h: white + space-time correlated; u,v white (hypothetical observation)")
# classical reference carried from the twin notebook (stage F for L1, stage B for L2; arm 'same', ramp 3 h, early window, U0=1, h and huv; recounted by the author from the JSON files twin2_results_standard_L2_stage[B,F].json)
CLASSICAL_REF = {("L1", 1.0, "h"): (10, 10), ("L1", 1.0, "huv"): (10, 10), ("L2", 1.0, "h"): (2, 10), ("L2", 1.0, "huv"): (6, 10)}

PREREG_P = dict(
    version="pinn-prereg-1", noise=_NOISE_PRE, classical_reference=[[list(k), list(v)] for k, v in CLASSICAL_REF.items()],
    question="With the SAME four-parameter zonal parameterisation, data windows, noise realisations (common random numbers) and identification rule as the classical arm, does a physics-informed neural network (Adam only) identify the friction structure from sea-level data at noise level L1, and does it do so as often as the classical arm?",
    data="PINN rows use data from the independent generator (arm 'indep': continuous-equation truth on a 4x finer grid, RK4) so that neither arm benefits from an inverse crime of the PINN's continuous equations; the classical reference (10/10) is arm 'same' (inverse crime) and the classical control (stage K) is arm 'indep'",
    gates="G1 autodiff derivative = finite difference (rel. err <= 2e-2); G2 PDE residual operator on generator fields: rms normalised residual of eta- and u-equations <= 0.05 and larger by >= 1.5x for a friction field multiplied by 2 (equation/sign/scale check); G3 forward test (theta true, PDE only, no data, hard BC/IC): rms(h error) <= 0.5 sigma_h_total, rms(u error) <= 0.5 sigma_vel, relative rms(v error) <= 0.5 (no momentum collapse), all at the run level on the window grid; G4 positive control (start from the forward solution, theta free, data of realisation 0): identified (25% / 6.25 km). A failed gate blocks the inverse rows.",
    identified="a PINN run is VALID if it is finite, its whitened data misfit <= chi2_ideal + 5*sqrt(2/n), rms(u_pred) >= 0.2*U0 and rms(v_pred) >= 0.0025*U0 (no momentum collapse); realisation is IDENTIFIED if every run equivalent to the best valid run (n*(chi2-best) <= 13.3) has Cf within 25% (both values) and mean boundary error <= 6.25 km; row IDENTIFIED/NOT_IDENTIFIED/MARGINAL by the 90%/10% rule of the twin notebook; chi2_ideal = whitened misfit of the true noise realisation",
    P1="PINN comparable to classical if the 95% Clopper-Pearson interval of the PINN count (level L1, U0=1) overlaps the interval of the classical reference (10/10, i.e. [0.69,1]); 'PINN worse' if the intervals are disjoint with the PINN below",
    P2="adding velocity helps the PINN if the lower CP bound of identified(huv) exceeds the upper CP bound of identified(h)",
    P3="decision table: classical reference identified & PINN not -> PINN optimisation problem (hypothesis 2 refuted); both identified -> field identifiable at this level; neither -> information limit",
    honesty="the classical reference was seen before this notebook was written; thresholds are fixed here, before any PINN inverse row, and hyper-parameters are tuned only on the forward test; 10 realisations are a count, not statistics",
)
PREREG_HASH = hashlib.sha256(json.dumps(PREREG_P, sort_keys=True).encode()).hexdigest()[:16]

_MODES = {
    "quick":    dict(U0_E=[1.0], U0_P=[1.0], N_STARTS=3, ZONAL_MAXITER=60, N_SEEDS=2, W0_CAP_H=40.0, N_REAL=2, N_INIT=1, GATE_CELLS=1),
    "standard": dict(U0_E=[1.0], U0_P=[1.0], N_STARTS=6, ZONAL_MAXITER=120, N_SEEDS=10, W0_CAP_H=96.0, N_REAL=10, N_INIT=3, GATE_CELLS=1),
}
CFG = dict(N_STEPS=1500, T_STRIDE=25, WIN_LEN=1350, P_OBS=39, SEED=7, CF_SCALE=2.5e-3,
           TOL_BND_KM=6.25, TOL_CF_REL=0.25, DCHI2_CRIT=13.3, SS_THRESH=0.05, NOISE_LEVEL=NOISE_LEVEL, WEIGHTING=WEIGHTING)
CFG.update(_MODES[MODE])
REGIMES = {}
GEN_R = 4

OUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
OUT_PATH = os.path.join(OUT_DIR, f"pinn_results_{MODE}_{NOISE_LEVEL}.json")
SNAP_TAG = f"{MODE}_{NOISE_LEVEL}_stage{STAGE}"
_KEYS = ("gate", "G", "G_attempts", "P", "K")
RESULTS = dict(mode=MODE, prereg=PREREG_P, prereg_hash=PREREG_HASH, pinn=PINN, pinn_hash=PINN_HASH, have_jax=HAVE_JAX, cfg=CFG,
               noise_levels=NOISE_LEVELS, noise_corr=NOISE_CORR, gate={}, G={}, G_attempts=[], P={}, K={}, E1={})
_resume_src = [p for p in [OUT_PATH] + sorted(glob.glob(f"/kaggle/input/**/pinn_results_{MODE}_{NOISE_LEVEL}*.json", recursive=True)) if os.path.exists(p)]
if RESUME and _resume_src:
    _old = json.load(open(_resume_src[0]))
    if _old.get("prereg_hash") == PREREG_HASH and _old.get("mode") == MODE:
        for k in _KEYS: RESULTS[k] = _old.get(k, RESULTS[k])
        print("RESUMED from", _resume_src[0])
    else:
        print("WARNING: existing results file", _resume_src[0], "has a different pre-registration hash / mode -> NOT resumed.")

def _jsonable(o):
    if isinstance(o, np.floating): return float(o)
    if isinstance(o, np.integer): return int(o)
    if isinstance(o, np.bool_): return bool(o)
    if isinstance(o, np.ndarray): return o.tolist()
    raise TypeError(type(o))
def save_results():
    with open(OUT_PATH, "w") as f: json.dump(RESULTS, f, default=_jsonable)

print("MODE:", MODE, "| STAGE:", STAGE, "| level:", NOISE_LEVEL, "-", NOISE_LEVELS[NOISE_LEVEL]["desc"], "| weighting:", WEIGHTING, "| PINN data arm:", ARM_PINN)
print("RUN:", RUN, "| output file:", OUT_PATH)
print("JAX:", HAVE_JAX, "| devices:", (jax.devices() if HAVE_JAX else "n/a"))
print("HASHES  pre-registration:", PREREG_HASH, "| PINN hyper-parameters:", PINN_HASH)
print("PINN:", PINN)


MODE: standard | STAGE: G | level: L1 - optimistic: 2 cm white SLA, 1 cm/s velocity | weighting: gls | PINN data arm: indep
RUN: {'GATE': True, 'E6': True} | output file: /kaggle/working/pinn_results_standard_L1.json
JAX: True | devices: [CudaDevice(id=0), CudaDevice(id=1)]
HASHES  pre-registration: b907f835f7a4fb04 | PINN hyper-parameters: 4515e1ed182e5d70
PINN: {'width': 128, 'depth': 5, 'n_harm': 3, 'steps': 40000, 'block': 500, 'n_col': 8192, 'lr0': 0.001, 'lr1': 2e-05, 'warm': 500, 'lam_p0': 1.0, 'lam_p1': 1000.0, 'f_lam': 0.6, 'f_freeze': 0.25, 'f_anneal': 0.7, 'lr_theta': 0.003, 'w_start': 0.3, 'lam_v': 1.0, 'v_scale': 'natural', 'hs_factor': 3.0, 'us_factor': 1.0, 'vs_factor': 0.01, 't_g_h': 1.0, 'net_seed': 0, 'ctrl_steps_frac': 0.5, 'ctrl_lr0': 0.0002}


## Solver, problem definition and observation-noise machinery (verbatim from the twin notebook)

In [2]:
# ===== solver + problem definition (physics identical to the earlier notebook; new: observation-noise machinery) =====
# ---- core physics + problem definition (xp-generic: numpy for truth/tests, jax.numpy for inversion) ----
LX, LY = 150e3, 50e3
NX, NY = 24, 12
DX, DY = LX / NX, LY / NY
G, H0, F0, DT = 9.81, 80.0, 6.5e-5, 60.0
OMEGA = 2 * np.pi / (12.42 * 3600.0)
RAMP_TR = RAMP_HOURS * 3600.0
def set_ramp(h):
    """change the smooth-start duration (0 = v3 cold start)."""
    global RAMP_TR, RAMP_HOURS
    RAMP_HOURS = h; RAMP_TR = h * 3600.0
def _ramp(xp, t):
    """smooth start of the boundary forcing: 0.5*(1-cos(pi t/TR)) for t < TR, then 1 (C1-continuous). RAMP_HOURS = 0 -> v3 behaviour."""
    if RAMP_TR <= 0: return 1.0
    return xp.where(t < RAMP_TR, 0.5 * (1.0 - xp.cos(xp.pi * t / RAMP_TR)), 1.0)
CF_LOW_TRUE, CF_HIGH_TRUE = 1.5e-3, 4.0e-3
B0_TRUE, S_TRUE = 0.5, 0.1          # boundary x/LX = B0 + S*(y/LY - 0.5)  (identical to the phase-0 twin)
XC = (np.arange(NX) + 0.5) * DX / LX     # normalized cell-centre coordinates
YC = (np.arange(NY) + 0.5) * DY / LY

def swe_step(xp, state, t, Cf, U0):
    eta, u, v = state
    vp = xp.pad(v, ((1, 1), (0, 0)), mode="edge"); vc = 0.5 * (vp[:, :-1] + vp[:, 1:]); v_at_u = 0.5 * (vc[:-1, :] + vc[1:, :])
    uc = 0.5 * (u[:-1, :] + u[1:, :]); up = xp.pad(uc, ((0, 0), (1, 1)), mode="edge"); u_at_v = 0.5 * (up[:, :-1] + up[:, 1:])
    Cf_u = Cf[np.clip(np.arange(NX + 1), 0, NX - 1), :]
    Cf_v = Cf[:, np.clip(np.arange(NY + 1), 0, NY - 1)]
    su = xp.sqrt(u ** 2 + v_at_u ** 2 + 1e-8); sv = xp.sqrt(u_at_v ** 2 + v ** 2 + 1e-8)
    de_dx = (eta[1:, :] - eta[:-1, :]) / DX; de_dy = (eta[:, 1:] - eta[:, :-1]) / DY
    fu = Cf_u / H0 * u * su; fv = Cf_v / H0 * v * sv
    un = u[1:-1, :] + DT * (-G * de_dx + F0 * v_at_u[1:-1, :] - fu[1:-1, :])
    u_new = xp.concatenate([xp.full((1, NY), U0 * _ramp(xp, t) * xp.cos(OMEGA * t)), un, xp.zeros((1, NY))], axis=0)
    vn = v[:, 1:-1] + DT * (-G * de_dy - F0 * u_at_v[:, 1:-1] - fv[:, 1:-1])
    v_new = xp.concatenate([xp.zeros((NX, 1)), vn, xp.zeros((NX, 1))], axis=1)
    div = (u_new[1:, :] - u_new[:-1, :]) / DX + (v_new[:, 1:] - v_new[:, :-1]) / DY
    return (eta - DT * H0 * div, u_new, v_new)

def run_obs_np(Cf, U0, n_steps, ox, oy):
    """numpy reference run -> (h,u,v) at observed cells for every step: 3 arrays (n_steps, P)."""
    s = (np.zeros((NX, NY)), np.zeros((NX + 1, NY)), np.zeros((NX, NY + 1)))
    H, U, V = [], [], []
    for n in range(n_steps):
        s = swe_step(np, s, n * DT, Cf, U0)
        e, u, v = s
        H.append(e[ox, oy]); U.append((0.5 * (u[:-1] + u[1:]))[ox, oy]); V.append((0.5 * (v[:, :-1] + v[:, 1:]))[ox, oy])
    return np.array(H), np.array(U), np.array(V)

def cf_zonal(xp, p, w_cells=0.5):
    """p = [ln Cf_low, ln Cf_high, b0, s]; smooth two-zone field on the cell grid (width = w_cells cells)."""
    X, Y = np.meshgrid(XC, YC, indexing="ij")
    arg = (X - (p[2] + p[3] * (Y - 0.5))) / (w_cells * DX / LX)
    sig = 1.0 / (1.0 + xp.exp(-arg))
    lo, hi = xp.exp(p[0]), xp.exp(p[1])
    return lo + (hi - lo) * sig

def cf_true_field():
    X, Y = np.meshgrid(XC, YC, indexing="ij")
    return np.where(X < B0_TRUE + S_TRUE * (Y - 0.5), CF_LOW_TRUE, CF_HIGH_TRUE)


# ---- K-zone model: K strips along x separated by K-1 parallel (common-slope) smooth boundaries ----
def cf_zonal_K(xp, theta, K, w_cells=0.5):
    """theta = [ln c_1..ln c_K, b_1..b_{K-1}, s]  (K=1: [ln c_1]).  Boundary k: x/LX = b_k + s*(y/LY-0.5)."""
    X, Y = np.meshgrid(XC, YC, indexing="ij")
    f = xp.exp(theta[0]) * xp.ones((NX, NY))
    for k in range(K - 1):
        arg = (X - (theta[K + k] + theta[2 * K - 1] * (Y - 0.5))) / (w_cells * DX / LX)
        f = f + (xp.exp(theta[k + 1]) - xp.exp(theta[k])) / (1.0 + xp.exp(-arg))
    return f



# =====================================================================================================================
# OBSERVATION-NOISE MACHINERY (NumPy; independent of JAX)
# =====================================================================================================================
# Covariance of the sea-level noise on a (time T x cells P) window:  C = sig_w^2 I + sig_c^2 (Kt (x) Ks)
#   Ks = Gaussian kernel on cell-centre distances (length L_space), Kt = exp(-|dt|/tau) on the actual snapshot times.
# Eigen-decompositions Ks = Us diag(ls) Us^T and Kt = Ut diag(lt) Ut^T diagonalise C exactly, so
#   draw:    e = sig_w z + sig_c (Ut sqrt(lt)) Z (Us sqrt(ls))^T     (z, Z standard normal T x P)
#   whiten:  r_w = (Ut^T r Us) / sqrt(sig_w^2 + sig_c^2 lt_i ls_j)   ->  E||r_w||^2 / (T P) = 1 for r = e  (exact GLS misfit)
# weighting 'white' instead divides by the total sd sqrt(sig_w^2 + sig_c^2) and ignores the correlation (mis-specified covariance).
def noise_spec(level=None):
    return NOISE_LEVELS[NOISE["level"] if level is None else level]

def _kernels(ox, oy, t_idx):
    xs = (np.asarray(ox, float) + 0.5) * DX; ys = (np.asarray(oy, float) + 0.5) * DY
    d2 = (xs[:, None] - xs[None, :]) ** 2 + (ys[:, None] - ys[None, :]) ** 2
    Ks = np.exp(-d2 / (2.0 * (NOISE_CORR["L_space_km"] * 1e3) ** 2))
    tt = np.asarray(t_idx, float) * DT / 3600.0
    Kt = np.exp(-np.abs(tt[:, None] - tt[None, :]) / NOISE_CORR["tau_time_h"])
    return Ks, Kt

_NZ_CACHE = {}
def noise_ctx(ox, oy, t_idx, level=None, weighting=None):
    """everything needed to draw and to whiten the noise of one window/layout at one level."""
    sp = noise_spec(level); weighting = WEIGHTING if weighting is None else weighting
    key = (tuple(int(a) for a in ox), tuple(int(a) for a in oy), tuple(int(a) for a in t_idx))
    if key not in _NZ_CACHE:
        Ks, Kt = _kernels(ox, oy, t_idx)
        ls, Us = np.linalg.eigh(Ks); lt, Ut = np.linalg.eigh(Kt)
        _NZ_CACHE[key] = (Us, np.clip(ls, 0.0, None), Ut, np.clip(lt, 0.0, None))
    Us, ls, Ut, lt = _NZ_CACHE[key]
    sw, sc = sp["sig_h_white"], sp["sig_h_corr"]
    var = sw ** 2 + sc ** 2 * (lt[:, None] * ls[None, :])
    inv_gls = 1.0 / np.sqrt(var)
    tot = math.sqrt(sw ** 2 + sc ** 2)
    ctx = dict(Us=Us, ls=ls, Ut=Ut, lt=lt, sig_h_white=sw, sig_h_corr=sc, sig_vel=sp["sig_vel"], sig_h_total=tot, inv_gls=inv_gls,
               weighting=weighting, level=(NOISE["level"] if level is None else level))
    if weighting == "gls":
        ctx.update(Wt=Ut, Ws=Us, inv=inv_gls)
    else:
        ctx.update(Wt=None, Ws=None, inv=np.full_like(inv_gls, 1.0 / tot))
    return ctx

def draw_noise(ctx, T, P, noise_seed, variant):
    """common random numbers: the same standard-normal draws are scaled by the sd of every level."""
    rng_w = np.random.default_rng(11 + noise_seed); rng_c = np.random.default_rng(5011 + noise_seed)
    e = dict(h=ctx["sig_h_white"] * rng_w.normal(size=(T, P)))
    Zc = rng_c.normal(size=(T, P))
    if ctx["sig_h_corr"] > 0:
        e["h"] = e["h"] + ctx["sig_h_corr"] * ((ctx["Ut"] * np.sqrt(ctx["lt"])) @ Zc @ (ctx["Us"] * np.sqrt(ctx["ls"])).T)
    if variant == "huv":
        e["u"] = ctx["sig_vel"] * rng_w.normal(size=(T, P)); e["v"] = ctx["sig_vel"] * rng_w.normal(size=(T, P))
    return e

def make_obs(clean, ox, oy, t_idx, variant, noise_seed, level=None):
    ctx = noise_ctx(ox, oy, t_idx, level)
    T, P = clean["h"].shape
    e = draw_noise(ctx, T, P, noise_seed, variant)
    return ctx, {k: clean[k] + e[k] for k in e}

def make_chi2(fwd, t_idx, obs, ctx, variant, xp=None):
    """noise-normalised misfit of a forward model fwd(Cf)->(H,U,V) (series at the observed cells): mean of squared whitened residuals.
    h: exact GLS whitening (or white weights), u,v: white. xp = jax.numpy (default) or numpy (tests)."""
    xp = jnp if xp is None else xp
    tj = xp.asarray(np.asarray(t_idx)); inv = xp.asarray(ctx["inv"]); sv = ctx["sig_vel"]
    if ctx["Wt"] is not None: Wt = xp.asarray(ctx["Wt"]); Ws = xp.asarray(ctx["Ws"])
    def chi2(Cf):
        H, U, V = fwd(Cf)
        r = H[tj] - obs["h"]
        if ctx["Wt"] is not None: r = (Wt.T @ r) @ Ws
        c = xp.mean((r * inv) ** 2)
        if variant == "huv":
            c = (c + xp.mean(((U[tj] - obs["u"]) / sv) ** 2) + xp.mean(((V[tj] - obs["v"]) / sv) ** 2)) / 3.0
        return c
    return chi2


CF_T = cf_true_field()
BOUNDS = [(np.log(5e-4), np.log(2e-2))] * 2 + [(0.1, 0.9), (-0.5, 0.5)]
P_TRUE = np.array([np.log(CF_LOW_TRUE), np.log(CF_HIGH_TRUE), B0_TRUE, S_TRUE])
if HAVE_JAX:
    # ---- JAX forward model returning only the observed cells (keeps memory tiny) ----
    def make_forward(U0, ox, oy):
        oxj, oyj = jnp.array(ox), jnp.array(oy)
        def fwd(Cf):
            def body(state, n):
                new = swe_step(jnp, state, n * DT, Cf, U0)
                e, u, v = new
                uc = 0.5 * (u[:-1, :] + u[1:, :]); vc = 0.5 * (v[:, :-1] + v[:, 1:])
                return new, (e[oxj, oyj], uc[oxj, oyj], vc[oxj, oyj])
            init = (jnp.zeros((NX, NY)), jnp.zeros((NX + 1, NY)), jnp.zeros((NX, NY + 1)))
            _, ys = lax.scan(body, init, jnp.arange(CFG["N_STEPS"]))
            return ys
        return fwd

## Tools: data windows, linear information, independent generator, classical rows (verbatim)

In [3]:
# ===================== tools: data windows, transient analysis, linear information, independent generator, classical rows =====================
def regime_idx(W0, regime):
    """snapshot steps of a data window whose first snapshot is at step W0 (W0=150 == v3f window)."""
    if regime == "rich":    return np.arange(W0, W0 + CFG["WIN_LEN"], CFG["T_STRIDE"])
    if regime == "sparse4": return W0 + np.array([225, 599, 974, 1349])
    raise ValueError(regime)

def set_window(W0):
    """set N_STEPS and REGIMES for a data window starting at step W0 (used by the v3f problem builders, which read these globals)."""
    CFG["N_STEPS"] = int(W0 + CFG["WIN_LEN"])
    REGIMES["rich"] = regime_idx(W0, "rich"); REGIMES["sparse4"] = regime_idx(W0, "sparse4")
set_window(150)

def cp_interval(k, n, alpha=0.05):
    """Clopper-Pearson interval for k successes out of n."""
    lo = 0.0 if k == 0 else stats.beta.ppf(alpha / 2, k, n - k + 1)
    hi = 1.0 if k == n else stats.beta.ppf(1 - alpha / 2, k + 1, n - k)
    return float(lo), float(hi)

# ---------------- E1: transient decay (NumPy only) ----------------
P_STEPS = 2 * np.pi / OMEGA / DT           # M2 period in steps (non-integer)

def run_full_np(Cf, U0, n):
    s = (np.zeros((NX, NY)), np.zeros((NX + 1, NY)), np.zeros((NX, NY + 1)))
    H = np.empty((n, NX, NY))
    for k in range(n):
        s = swe_step(np, s, k * DT, Cf, U0); H[k] = s[0]
    return H

def periodicity_defect(H):
    """list of (block end time [h], d_k) for blocks k = 1, 2, ... of one M2 period (block k compared with the same phase one period earlier)."""
    out, k = [], 1
    while (k + 1) * P_STEPS <= len(H):
        idx = np.arange(int(k * P_STEPS), int((k + 1) * P_STEPS))
        prev = idx - P_STEPS; lo = np.floor(prev).astype(int); fr = prev - lo
        hp = H[lo] * (1 - fr)[:, None, None] + H[lo + 1] * fr[:, None, None]
        d = np.sqrt(np.mean((H[idx] - hp) ** 2)) / np.sqrt(np.mean(H[idx] ** 2))
        out.append(((k + 1) * P_STEPS * DT / 3600.0, float(d))); k += 1
    return out

def steady_time(blocks, thr):
    """end time [h] of the first block after which every block has d < thr; None if never."""
    for i in range(len(blocks)):
        if all(b[1] < thr for b in blocks[i:]): return blocks[i][0]
    return None

def residual_period(H, cell=None, n_harm=4):
    """dominant period [h] and relative std of the residual of one mid-channel cell over t in [P, 3P] after fitting const + trend + M2 and its overtones."""
    cell = (NX // 2, NY // 2) if cell is None else cell
    a, b = int(P_STEPS), min(int(3 * P_STEPS), len(H))
    t = np.arange(a, b) * DT; y = H[a:b, cell[0], cell[1]]
    cols = [np.ones_like(t), t - t.mean()]
    for m in range(1, n_harm + 1): cols += [np.cos(m * OMEGA * t), np.sin(m * OMEGA * t)]
    A = np.stack(cols, 1); coef = np.linalg.lstsq(A, y, rcond=None)[0]; r = y - A @ coef
    rw = r * np.hanning(len(r)); nfft = 16 * len(r)
    sp = np.abs(np.fft.rfft(rw, nfft)); f = np.fft.rfftfreq(nfft, DT)
    i = int(np.argmax(sp[1:])) + 1
    return float(1.0 / f[i] / 3600.0), float(np.std(r) / np.std(y))

# ---------------- E2: linearised information ----------------
def layout_cells(layout):
    rng = np.random.default_rng(CFG["SEED"] + layout)
    cells = rng.choice(NX * NY, CFG["P_OBS"], replace=False)
    return np.unravel_index(cells, (NX, NY))

def jac_fd(U0, ox, oy, n, h=1e-4):
    """NumPy central finite differences of the observed (h,u,v) series w.r.t. the 4 zonal parameters at P_TRUE (smooth truth)."""
    base = run_obs_np(cf_zonal(np, P_TRUE), U0, n, ox, oy); cols = []
    for i in range(4):
        pp = P_TRUE.copy(); pp[i] += h; pm = P_TRUE.copy(); pm[i] -= h
        a = run_obs_np(cf_zonal(np, pp), U0, n, ox, oy); b = run_obs_np(cf_zonal(np, pm), U0, n, ox, oy)
        cols.append([(x - y) / (2 * h) for x, y in zip(a, b)])
    return list(base), [np.stack([cols[i][v] for i in range(4)], -1) for v in range(3)]

if HAVE_JAX:
    def jac_jax(U0, ox, oy, n):
        old = CFG["N_STEPS"]; CFG["N_STEPS"] = n
        try:
            fwd = make_forward(U0, ox, oy)
            g = lambda p: fwd(cf_zonal(jnp, p))
            both = jax.jit(lambda p: (g(p), jax.jacfwd(g)(p)))
            base, J = both(jnp.array(P_TRUE))
        finally:
            CFG["N_STEPS"] = old
        return [np.asarray(b) for b in base], [np.asarray(j) for j in J]

def fisher_metrics(J, t_idx, variant, ox, oy, level):
    """linearised uncertainty of (ln Cf_low, ln Cf_high, boundary offset b0, tilt s) for the window t_idx at one noise level, with the exact
    space-time noise covariance of that level (GLS-whitened Jacobian). h-only: all P_OBS cells; huv: first P_OBS//3 cells, h+u+v (u, v white)."""
    Pu = CFG["P_OBS"] if variant == "h" else CFG["P_OBS"] // 3
    ctx = noise_ctx(np.asarray(ox)[:Pu], np.asarray(oy)[:Pu], t_idx, level, weighting="gls")
    Jh = J[0][t_idx][:, :Pu, :]
    W = np.einsum("ta,tpk,pb->abk", ctx["Ut"], Jh, ctx["Us"]) * ctx["inv_gls"][:, :, None]
    rows = [W.reshape(-1, 4)]
    if variant == "huv":
        for vi in (1, 2): rows.append(J[vi][t_idx][:, :Pu, :].reshape(-1, 4) / ctx["sig_vel"])
    Jt = np.vstack(rows); F = Jt.T @ Jt
    sv = np.linalg.svd(Jt, compute_uv=False)
    C = np.linalg.pinv(F); sg = np.sqrt(np.clip(np.diag(C), 0, None))
    return dict(sd_lnClow=float(sg[0]), sd_lnChigh=float(sg[1]), sd_b0_km=float(sg[2] * LX / 1e3), sd_tilt_km=float(sg[3] * LX / 1e3),
                sd_bnd_km=float(LX / 1e3 * math.sqrt(C[2, 2] + C[3, 3] / 12.0)), sv=[float(x) for x in sv], cond_F=float(np.linalg.cond(F)))

# ---------------- independent data generator (NumPy; same equations, different numerics) ----------------
# ======================= independent generator =======================
# Same continuous equations as the inversion solver (no advection, no viscosity, quadratic friction, f-plane, same ramp),
# DIFFERENT numerics: r-times finer grid, classical RK4 in time, truth front area-averaged on the fine cells, face friction = mean of the two neighbouring cells,
# boundary forcing evaluated at the true time. Truth Cf is evaluated at the fine cell centres (sharp step on the fine grid).
def _grid(r):
    nx, ny = NX * r, NY * r
    return nx, ny, LX / nx, LY / ny, (np.arange(nx) + 0.5) / nx, (np.arange(ny) + 0.5) / ny

def _rhs(eta, u, v, t, Cf_u, Cf_v, U0, dx, dy):
    u = u.copy(); v = v.copy()
    u[0, :] = U0 * float(_ramp(np, t)) * np.cos(OMEGA * t); u[-1, :] = 0.0
    v[:, 0] = 0.0; v[:, -1] = 0.0
    v_c = 0.5 * (v[:, :-1] + v[:, 1:]); u_c = 0.5 * (u[:-1, :] + u[1:, :])
    v_at_u = np.zeros_like(u); v_at_u[1:-1, :] = 0.5 * (v_c[:-1, :] + v_c[1:, :])
    u_at_v = np.zeros_like(v); u_at_v[:, 1:-1] = 0.5 * (u_c[:, :-1] + u_c[:, 1:])
    su = np.sqrt(u ** 2 + v_at_u ** 2 + 1e-8); sv = np.sqrt(u_at_v ** 2 + v ** 2 + 1e-8)
    du = np.zeros_like(u); dv = np.zeros_like(v)
    du[1:-1, :] = -G * (eta[1:, :] - eta[:-1, :]) / dx + F0 * v_at_u[1:-1, :] - Cf_u[1:-1, :] / H0 * u[1:-1, :] * su[1:-1, :]
    dv[:, 1:-1] = -G * (eta[:, 1:] - eta[:, :-1]) / dy - F0 * u_at_v[:, 1:-1] - Cf_v[:, 1:-1] / H0 * v[:, 1:-1] * sv[:, 1:-1]
    deta = -H0 * ((u[1:, :] - u[:-1, :]) / dx + (v[:, 1:] - v[:, :-1]) / dy)
    return deta, du, dv

def _block_mean(a, r):
    return a.reshape(NX, r, NY, r).mean(axis=(1, 3))

def run_generator(cf_fn, U0, r=2, n_steps=None, rec_volume=False):
    """(h, u, v) at COARSE cell centres, shape (n_steps, NX, NY); entry n = state at time (n+1)*DT, exactly like run_obs_np
    and the JAX forward model. Fine fields are block-averaged r x r onto the coarse cells."""
    n_steps = CFG["N_STEPS"] if n_steps is None else n_steps
    nx, ny, dx, dy, xc, yc = _grid(r)
    X, Y = np.meshgrid(xc, yc, indexing="ij")
    Cf = cf_fn(X, Y)
    Cf_u = np.zeros((nx + 1, ny)); Cf_u[1:-1, :] = 0.5 * (Cf[:-1, :] + Cf[1:, :])
    Cf_v = np.zeros((nx, ny + 1)); Cf_v[:, 1:-1] = 0.5 * (Cf[:, :-1] + Cf[:, 1:])
    eta = np.zeros((nx, ny)); u = np.zeros((nx + 1, ny)); v = np.zeros((nx, ny + 1))
    dt = DT / r
    Hh = np.empty((n_steps, NX, NY)); Uu = np.empty_like(Hh); Vv = np.empty_like(Hh); vol = np.empty(n_steps)
    f = lambda e, uu, vv, t: _rhs(e, uu, vv, t, Cf_u, Cf_v, U0, dx, dy)
    for k in range(1, n_steps * r + 1):
        t = (k - 1) * dt
        k1 = f(eta, u, v, t)
        k2 = f(eta + 0.5 * dt * k1[0], u + 0.5 * dt * k1[1], v + 0.5 * dt * k1[2], t + 0.5 * dt)
        k3 = f(eta + 0.5 * dt * k2[0], u + 0.5 * dt * k2[1], v + 0.5 * dt * k2[2], t + 0.5 * dt)
        k4 = f(eta + dt * k3[0], u + dt * k3[1], v + dt * k3[2], t + dt)
        eta = eta + dt / 6.0 * (k1[0] + 2 * k2[0] + 2 * k3[0] + k4[0])
        u = u + dt / 6.0 * (k1[1] + 2 * k2[1] + 2 * k3[1] + k4[1])
        v = v + dt / 6.0 * (k1[2] + 2 * k2[2] + 2 * k3[2] + k4[2])
        if k % r == 0:
            n = k // r - 1
            u[0, :] = U0 * float(_ramp(np, k * dt)) * np.cos(OMEGA * k * dt); u[-1, :] = 0.0   # boundary faces hold the imposed value at the new time
            Hh[n] = _block_mean(eta, r)
            Uu[n] = _block_mean(0.5 * (u[:-1, :] + u[1:, :]), r)
            Vv[n] = _block_mean(0.5 * (v[:, :-1] + v[:, 1:]), r)
            vol[n] = eta.sum() * dx * dy
    return (Hh, Uu, Vv, vol) if rec_volume else (Hh, Uu, Vv)

_B = lambda X, Y: B0_TRUE + S_TRUE * (Y - 0.5)
TRUTH_FN_SHARP = {      # point-wise definitions (the coarse 'same' arm evaluates these at coarse cell centres = v3 truths)
    "standard":   lambda X, Y: np.where(X < _B(X, Y), CF_LOW_TRUE, CF_HIGH_TRUE),
    "gradient":   lambda X, Y: CF_LOW_TRUE + (CF_HIGH_TRUE - CF_LOW_TRUE) * X,
    "uniform":    lambda X, Y: np.full_like(X, 2.5e-3),
}
def _antialias(fn, s=8):
    """area-average of fn over each generator cell (s x s sub-samples): removes the staircase of a sharp tilted front, which otherwise
    makes the (tiny) transverse velocity v converge only at first order with refinement."""
    def g(X, Y):
        nx, ny = X.shape; off = (np.arange(s) + 0.5) / s - 0.5; acc = np.zeros_like(X)
        for a in off:
            for b in off: acc += fn(X + a / nx, Y + b / ny)
        return acc / (s * s)
    return g
TRUTH_FN = {k: _antialias(f) for k, f in TRUTH_FN_SHARP.items()}
_GEN_CACHE = {}
def gen_fields(truth, U0, r=None):
    r = GEN_R if r is None else r
    key = (truth, float(U0), int(r), CFG["N_STEPS"], RAMP_HOURS)
    if key not in _GEN_CACHE: _GEN_CACHE[key] = run_generator(TRUTH_FN[truth], U0, r)
    return _GEN_CACHE[key]


# ---------------- classical rows (arms 'same' and 'indep') ----------------
ARM = "same"
def _layout_for_seed(P_use, noise_seed):
    rng = np.random.default_rng(CFG["SEED"] + noise_seed)
    cells = rng.choice(NX * NY, CFG["P_OBS"], replace=False)
    return np.unravel_index(cells[:P_use], (NX, NY))

def build_problem_from_clean(U0, regime, variant, P_use, noise_seed, clean_fn, level=None):
    """common part of both arms: layout (same seeds), noise (common random numbers), forward model and misfit. clean_fn(ox, oy, t_idx) -> dict(h,u,v) of shape (T, P)."""
    ox, oy = _layout_for_seed(P_use, noise_seed)
    t_idx = REGIMES[regime]
    clean = clean_fn(ox, oy, t_idx)
    ctx, obs = make_obs(clean, ox, oy, t_idx, variant, noise_seed, level)
    fwd = make_forward(U0, ox, oy)
    return dict(U0=U0, regime=regime, variant=variant, P=P_use, ox=ox, oy=oy, t_idx=t_idx, obs=obs, ctx=ctx, fwd=fwd,
                chi2=make_chi2(fwd, t_idx, obs, ctx, variant), n_obs=int(obs["h"].size * (3 if variant == "huv" else 1)))

if HAVE_JAX:
    def build_problem(U0, regime, variant, P_use, noise_seed=0, cf_truth=None, level=None):
        if ARM == "same":      # truth from the NumPy loop of the SAME scheme as the inversion solver ('inverse crime')
            cf = CF_T if cf_truth is None else cf_truth
            def clean_fn(ox, oy, t_idx):
                Hc, Uc, Vc = run_obs_np(cf, U0, CFG["N_STEPS"], ox, oy)
                return dict(h=Hc[t_idx], u=Uc[t_idx], v=Vc[t_idx])
        else:                  # clean data from the independent generator
            if cf_truth is not None: raise ValueError("arm 'indep': only the standard sharp truth is wired")
            def clean_fn(ox, oy, t_idx):
                Hf, Uf, Vf = gen_fields("standard", U0)
                return dict(h=Hf[t_idx][:, ox, oy], u=Uf[t_idx][:, ox, oy], v=Vf[t_idx][:, ox, oy])
        return build_problem_from_clean(U0, regime, variant, P_use, noise_seed, clean_fn, level)

    def fit_zonal(pb, p0):
        vgf = jax.jit(jax.value_and_grad(lambda p: pb["chi2"](cf_zonal(jnp, p))))
        def f(p):
            v, g = vgf(jnp.array(p)); return float(v), np.asarray(g, dtype=np.float64)
        res = minimize(f, p0, jac=True, method="L-BFGS-B", bounds=BOUNDS, options=dict(maxiter=CFG["ZONAL_MAXITER"], ftol=1e-12, gtol=1e-8))
        p = res.x
        off = float((p[2] - B0_TRUE) * LX / 1e3); tilt = float((p[3] - S_TRUE) * LX / 1e3)
        return dict(cf_low=float(np.exp(p[0])), cf_high=float(np.exp(p[1])), b0=float(p[2]), s=float(p[3]), chi2=float(res.fun), nit=int(res.nit),
                    bnd_km=float(np.mean(np.abs((p[2] - B0_TRUE) + (p[3] - S_TRUE) * (YC - 0.5))) * LX / 1e3),
                    off_km=off, tilt_km=tilt, edge_km=float(max(abs(off - tilt / 2), abs(off + tilt / 2))))

    def zonal_one_seed(U0, regime, variant, P_use, ns, cf_truth=None, level=None):
        pbz = build_problem(U0, regime, variant, P_use, noise_seed=ns, cf_truth=cf_truth, level=level)
        chi2_ref = float(pbz["chi2"](cf_zonal(jnp, jnp.array(P_TRUE))))
        sols = []
        for k in range(CFG["N_STARTS"]):
            r = np.random.default_rng(100 + k + 17 * ns)
            p0 = np.array([np.log(2.5e-3) + r.normal(0, 0.4), np.log(2.5e-3) + r.normal(0, 0.4), r.uniform(0.3, 0.7), r.uniform(-0.2, 0.2)])
            sols.append(fit_zonal(pbz, p0))
        return dict(noise_seed=ns, chi2_ref=chi2_ref, n_obs=pbz["n_obs"], sols=sols, level=pbz["ctx"]["level"], weighting=pbz["ctx"]["weighting"])

def classify_seed(sd, tol_cf=0.25, tol_bnd=6.25, use_edge=False):
    n = sd["n_obs"]; sols = sd["sols"]; best = min(s["chi2"] for s in sols)
    if best > sd["chi2_ref"] + 2 * np.sqrt(2 / n): return "optimiser_failed"
    equiv = [s for s in sols if n * (s["chi2"] - best) <= CFG["DCHI2_CRIT"]]
    key = "edge_km" if use_edge else "bnd_km"
    ok = all(s[key] <= tol_bnd and abs(s["cf_low"] / CF_LOW_TRUE - 1) <= tol_cf and abs(s["cf_high"] / CF_HIGH_TRUE - 1) <= tol_cf for s in equiv)
    return "identified" if ok else "not_identified"

def misspec_flag(sd): return sd["chi2_ref"] > 1 + 5 * np.sqrt(2 / sd["n_obs"])

def classify_row(per_seed, tol_cf=0.25, tol_bnd=6.25, use_edge=False):
    labs = [classify_seed(sd, tol_cf, tol_bnd, use_edge) for sd in per_seed]
    ni, nn, nf = labs.count("identified"), labs.count("not_identified"), labs.count("optimiser_failed")
    valid = ni + nn
    if valid < 0.8 * len(per_seed) or valid == 0: cls = "INVALID"
    elif ni >= math.ceil(0.9 * valid): cls = "IDENTIFIED"
    elif ni <= math.floor(0.1 * valid): cls = "NOT_IDENTIFIED"
    else: cls = "MARGINAL"
    return dict(identified=ni, not_identified=nn, optimiser_failed=nf, n_seeds=len(per_seed), cls=cls)

def late_W0(U0):
    """first window start (steps, multiple of the stride) at or after the steady time t_ss(U0) from E1, clipped to [150, W0_CAP]."""
    cap = int(CFG["W0_CAP_H"] * 3600 / DT)
    e1 = RESULTS["E1"].get(str(U0), {})
    t_ss = e1.get("t_ss_h")
    flag = "ok"
    if t_ss is None:
        W0 = cap; flag = "steady state NOT reached in E1 run -> cap"
    else:
        W0 = int(math.ceil(t_ss * 3600 / DT / CFG["T_STRIDE"]) * CFG["T_STRIDE"])
        if W0 > cap: W0 = cap; flag = "t_ss beyond cap -> cap"
    return max(150, W0), flag

def run_row(store, tag, U0, variant, n_seeds, W0, cf_truth=None, regime="rich", arm="same", level=None):
    """run (or resume) one classical row for the window starting at step W0 at noise level `level` (default: baseline); results stored in store[tag]."""
    global ARM
    lvl = NOISE_LEVEL if level is None else level
    ARM = arm; set_window(W0); set_noise(lvl)
    P_use = CFG["P_OBS"] if variant == "h" else CFG["P_OBS"] // 3
    rows = store.setdefault(tag, dict(U0=U0, variant=variant, regime=regime, W0=int(W0), arm=arm, level=lvl, weighting=WEIGHTING, seeds=[]))["seeds"]
    t0 = time.time()
    for ns in range(len(rows), n_seeds):
        rows.append(zonal_one_seed(U0, regime, variant, P_use, ns, cf_truth=cf_truth, level=lvl)); save_results()
    c = classify_row(rows); lo, hi = cp_interval(c["identified"], c["n_seeds"])
    mc = float(np.mean([s["chi2_ref"] for s in rows]))
    print(f"[row] {tag:46s} W0={W0*DT/3600:6.1f} h  noise={lvl}  identified {c['identified']:2d}/{c['n_seeds']}  CP95=[{lo:.2f},{hi:.2f}]  -> {c['cls']:15s} chi2_ref(mean)={mc:5.2f}  flagged={sum(misspec_flag(s) for s in rows)}  [{time.time()-t0:.0f}s]")
    set_window(150); set_noise(NOISE_LEVEL); ARM = "same"
    return c

## GATE: forward model, Jacobian, noise-model self-test (verbatim; classical solver)

In [4]:
# ===================== GATE =====================

# ---- noise-model self-test (NumPy only): the draw and the whitening must be exactly consistent ----
def noise_selftest():
    ok = {}; rng = np.random.default_rng(0)
    cells = rng.choice(NX * NY, CFG["P_OBS"], replace=False); ox, oy = np.unravel_index(cells, (NX, NY))
    t_idx = regime_idx(150, "rich"); T, P = len(t_idx), len(ox)
    for lvl in ("L1", "L3"):
        ctx_g = noise_ctx(ox, oy, t_idx, lvl, "gls"); ctx_w = noise_ctx(ox, oy, t_idx, lvl, "white")
        E_ = np.stack([draw_noise(ctx_g, T, P, s, "h")["h"] for s in range(300)])          # (300, T, P)
        chi_g = np.mean([np.mean(((ctx_g["Ut"].T @ e @ ctx_g["Us"]) * ctx_g["inv"]) ** 2) for e in E_])
        chi_w = np.mean([np.mean((e * ctx_w["inv"]) ** 2) for e in E_])
        var_ratio = float(E_.var() / ctx_g["sig_h_total"] ** 2)
        ok[f"{lvl}_gls_chi2_mean"] = bool(abs(chi_g - 1.0) < 0.02)
        ok[f"{lvl}_white_chi2_mean"] = bool(abs(chi_w - 1.0) < 0.05)
        ok[f"{lvl}_total_variance"] = bool(abs(var_ratio - 1.0) < 0.05)
        print(f"   noise self-test {lvl}: mean whitened chi2 (gls) = {chi_g:.4f}  (white weights) = {chi_w:.4f}  variance / sd_total^2 = {var_ratio:.3f}")
    # sigma_c = 0 must reduce to plain division by sigma_w
    sp = NOISE_LEVELS["L1"]; ctx = noise_ctx(ox, oy, t_idx, "L1", "gls"); r = np.random.default_rng(1).normal(size=(T, P))
    a = (ctx["Ut"].T @ r @ ctx["Us"]) * ctx["inv"]; ok["white_limit_isometry"] = bool(np.allclose(np.sum(a ** 2), np.sum((r / sp["sig_h_white"]) ** 2), rtol=1e-9))
    # empirical spatial correlation of the correlated component vs the kernel (informational, loose tolerance)
    ctx3 = noise_ctx(ox, oy, t_idx, "L3", "gls"); Ks, Kt = _kernels(ox, oy, t_idx)
    Zs = [(ctx3["Ut"] * np.sqrt(ctx3["lt"])) @ np.random.default_rng(s).normal(size=(T, P)) @ (ctx3["Us"] * np.sqrt(ctx3["ls"])).T for s in range(300)]
    emp = np.mean([z.T @ z / T for z in Zs], axis=0); emp_c = emp / np.sqrt(np.outer(np.diag(emp), np.diag(emp)))
    dev = float(np.max(np.abs(emp_c - Ks)))
    ok["spatial_corr_matches_kernel"] = bool(dev < 0.12)
    print(f"   noise self-test: max |empirical - kernel| spatial correlation = {dev:.3f} (tolerance 0.12); sigma_c=0 isometry: {ok['white_limit_isometry']}")
    return bool(all(ok.values())), ok
t0 = time.time(); gate = {}
set_ramp(RAMP_DEFAULT)
if RUN["GATE"]:
    H = run_full_np(CF_T, 1.0, 3000)
    gate["numpy_long_run_finite"] = bool(np.isfinite(H).all() and np.abs(H).max() < 20.0)
    print(f"numpy long run finite and |h|max = {np.abs(H).max():.3f} m  -> {gate['numpy_long_run_finite']}")
    if HAVE_JAX:
        ox, oy = layout_cells(0); n = 300; errs = {}
        for rh in (3.0, 0.0):
            set_ramp(rh); old = CFG["N_STEPS"]; CFG["N_STEPS"] = n
            fwd = make_forward(1.0, ox, oy)
            yj = [np.asarray(a) for a in jax.jit(lambda: fwd(jnp.array(CF_T)))()]
            CFG["N_STEPS"] = old
            yn = run_obs_np(CF_T, 1.0, n, ox, oy)
            errs[rh] = max(float(np.max(np.abs(a - b)) / (np.std(b) + 1e-30)) for a, b in zip(yj, yn))
        set_ramp(RAMP_DEFAULT)
        gate["jax_vs_numpy_forward_max_rel"] = errs
        print("JAX vs NumPy forward model, max |diff|/std (ramp 3 h, ramp 0):", errs)
        nJ = 1500; ox, oy = layout_cells(0)
        bj, Jj = jac_jax(1.0, ox, oy, nJ); bf, Jf = jac_fd(1.0, ox, oy, nJ)
        jerr = [float(np.max(np.abs(a - b)) / (np.max(np.abs(b)) + 1e-30)) for a, b in zip(Jj, Jf)]
        gate["jacobian_jax_vs_fd_rel"] = jerr
        print("Jacobian JAX (jacfwd) vs NumPy central differences, max rel diff for h, u, v:", [f"{x:.2e}" for x in jerr])
        gate["passed"] = bool(gate["numpy_long_run_finite"] and all(e < 1e-8 for e in errs.values()) and all(e < 1e-3 for e in jerr))
    else:
        gate["passed"] = False
        print("JAX unavailable: only the NumPy check ran; E3/E4 will be skipped.")
    ok_n, det_n = noise_selftest()
    gate["noise_selftest"] = ok_n; gate["noise_selftest_detail"] = det_n
    gate["passed"] = bool(gate["passed"] and ok_n)
    RESULTS["gate"] = gate; save_results()
    print("GATE", "PASSED" if gate["passed"] else "FAILED", f"[{time.time()-t0:.0f}s]")
    if HAVE_JAX and not gate["passed"]:
        raise RuntimeError("GATE FAILED - do not interpret any later result; send this output to Claude.")

set_ramp(RAMP_DEFAULT)
assert RAMP_HOURS == RAMP_DEFAULT == 3.0, "ramp must be 3 h after the gate"
print("ramp restored to", RAMP_HOURS, "h")

numpy long run finite and |h|max = 6.782 m  -> True
JAX vs NumPy forward model, max |diff|/std (ramp 3 h, ramp 0): {3.0: 2.067331594027341e-13, 0.0: 1.2618516104024743e-13}
Jacobian JAX (jacfwd) vs NumPy central differences, max rel diff for h, u, v: ['2.00e-07', '7.76e-07', '1.86e-06']
   noise self-test L1: mean whitened chi2 (gls) = 0.9986  (white weights) = 0.9986  variance / sd_total^2 = 0.999
   noise self-test L3: mean whitened chi2 (gls) = 0.9984  (white weights) = 0.9930  variance / sd_total^2 = 0.993
   noise self-test: max |empirical - kernel| spatial correlation = 0.062 (tolerance 0.12); sigma_c=0 isometry: True
GATE PASSED [8s]
ramp restored to 3.0 h


## GATE: independent data generator (verbatim; needed because the PINN rows use the generator as truth)

In [5]:
# ===================== GATE (independent data generator) — runs only when stage E (or ALL) is requested =====================
if RUN["E6"]:
    gg = {}; _rms = lambda a: float(np.sqrt(np.mean(np.asarray(a) ** 2)))
    set_ramp(RAMP_DEFAULT); set_window(150); rich = REGIMES["rich"]
    t0 = time.time()
    # (a) exact volume balance: V(t) = H0*LY*U0*int_0^t ramp(s) cos(w s) ds (RK4 conserves this linear invariant to round-off)
    U0g = CFG["U0_E"][-1]
    _, _, _, vol = run_generator(TRUTH_FN["standard"], U0g, GEN_R, rec_volume=True)
    sfine = np.linspace(0.0, CFG["N_STEPS"] * DT, CFG["N_STEPS"] * 200 + 1)
    integrand = U0g * np.array([float(_ramp(np, s)) for s in sfine]) * np.cos(OMEGA * sfine)
    cum = np.concatenate([[0.0], np.cumsum(0.5 * (integrand[1:] + integrand[:-1]) * np.diff(sfine))])
    vexp = H0 * LY * cum[200::200][:CFG["N_STEPS"]]
    verr = np.max(np.abs(vol - vexp)) / np.max(np.abs(vexp))
    gg["generator_volume"] = bool(verr < 1e-6)
    print(f"(a) generator volume balance: max rel err = {verr:.2e} ->", "PASS" if gg["generator_volume"] else "FAIL")
    # (b) generator convergence: r in {1, GEN_R, 2*GEN_R}; reference = 2*GEN_R
    sig_min = [NOISE_LEVELS["L0"]["sig_h_white"] if NOISE_LEVELS["L0"]["sig_h_corr"] == 0 else math.hypot(NOISE_LEVELS["L0"]["sig_h_white"], NOISE_LEVELS["L0"]["sig_h_corr"]),
               NOISE_LEVELS["L0"]["sig_vel"], NOISE_LEVELS["L0"]["sig_vel"]]
    print(f"    convergence is judged against the SMALLEST noise sd of the notebook (level L0): h {sig_min[0]*100:.1f} cm, u/v {sig_min[1]*100:.2f} cm/s")
    for U0 in CFG["U0_E"]:
        F = {r: run_generator(TRUTH_FN["standard"], U0, r) for r in sorted({1, GEN_R, 2 * GEN_R})}
        ref = F[2 * GEN_R]
        for r in F:
            if r == 2 * GEN_R: continue
            e = [_rms(F[r][k][rich] - ref[k][rich]) / sig_min[k] for k in range(3)]
            tag = ""
            if r == GEN_R:
                okc = bool(max(e) <= 0.25); gg[f"generator_conv_U0={U0}"] = okc; tag = " <- GEN_R: " + ("PASS" if okc else "FAIL (use a larger GEN_R)")
            print(f"(b) U0={U0:<5} r={r}: rms(G_r - G_{2*GEN_R}) / sd_L0   h={e[0]:.3f}  u={e[1]:.3f}  v={e[2]:.3f}{tag}")
    # (c) size of the 'inverse crime' (reported, no pass/fail)
    ax_, ay_ = np.unravel_index(np.arange(NX * NY), (NX, NY)); Xc, Yc = np.meshgrid(XC, YC, indexing="ij"); Tc = TRUTH_FN_SHARP["standard"](Xc, Yc)
    spb = noise_spec(NOISE_LEVEL); sig_base = [math.hypot(spb["sig_h_white"], spb["sig_h_corr"]), spb["sig_vel"], spb["sig_vel"]]
    print(f"(c) model error = rms(inversion solver - generator): absolute, and relative to the BASELINE noise sd ({NOISE_LEVEL}: h {sig_base[0]*100:.1f} cm, u/v {sig_base[1]*100:.1f} cm/s)")
    for U0 in CFG["U0_E"]:
        Hc, Uc, Vc = [a.reshape(CFG["N_STEPS"], NX, NY) for a in run_obs_np(Tc, U0, CFG["N_STEPS"], ax_, ay_)]
        Gf = gen_fields("standard", U0)
        ea = [_rms(a[rich] - b[rich]) for a, b in zip((Hc, Uc, Vc), Gf)]
        print(f"      U0={U0:<5} rich: h={ea[0]*100:.3f} cm ({ea[0]/sig_base[0]:.3f} sd)  u={ea[1]*100:.3f} cm/s ({ea[1]/sig_base[1]:.3f} sd)  v={ea[2]*100:.3f} cm/s ({ea[2]/sig_base[2]:.3f} sd)")
        gg[f"model_error_U0={U0}"] = dict(h_m=ea[0], u_ms=ea[1], v_ms=ea[2], h_rel=ea[0]/sig_base[0], u_rel=ea[1]/sig_base[1], v_rel=ea[2]/sig_base[2])
    gg["generator_ok"] = bool(all(v for k, v in gg.items() if isinstance(v, bool)))
    RESULTS["gate"].update(gg)
    if HAVE_JAX: RESULTS["gate"]["passed"] = bool(RESULTS["gate"].get("passed") and gg["generator_ok"])
    save_results()
    print("GENERATOR GATE", "PASSED" if gg["generator_ok"] else "FAILED", f"[{time.time()-t0:.0f}s]")
    if not gg["generator_ok"]:
        raise RuntimeError("GENERATOR GATE FAILED - do not interpret any later result of stage E; send this output to Claude.")

(a) generator volume balance: max rel err = 8.97e-11 -> PASS
    convergence is judged against the SMALLEST noise sd of the notebook (level L0): h 1.0 cm, u/v 0.50 cm/s
(b) U0=1.0   r=1: rms(G_r - G_8) / sd_L0   h=0.492  u=0.351  v=0.134
(b) U0=1.0   r=4: rms(G_r - G_8) / sd_L0   h=0.031  u=0.022  v=0.010 <- GEN_R: PASS
(c) model error = rms(inversion solver - generator): absolute, and relative to the BASELINE noise sd (L1: h 2.0 cm, u/v 1.0 cm/s)
      U0=1.0   rich: h=1.405 cm (0.703 sd)  u=0.583 cm/s (0.583 sd)  v=0.192 cm/s (0.192 sd)
GENERATOR GATE PASSED [65s]


## PINN core: network with hard constraints, continuous residuals, data and whitened misfit (xp-generic, NumPy-tested)

In [6]:
# ===================== PINN core (xp-generic: NumPy for tests, jax.numpy for training) =====================
# Unknowns: eta, u, v at (x, y, t) from one MLP. Hard constraints: u(0,y,t) = U0 ramp(t) cos(wt) (imposed tide), u(LX,y,t) = 0, v(x,0,t) = v(x,LY,t) = 0,
# eta = u = v = 0 at t = 0 (the 3 h ramp makes the imposed boundary velocity vanish at t = 0). Equations = the continuous form of the solver's equations
# (no advection, no viscosity, quadratic friction with the same 1e-8 regularisation, f-plane).
class _K: pass
def pinn_consts(U0, pc=None):
    pc = PINN if pc is None else pc
    assert RAMP_HOURS > 0, "the PINN needs the smooth start (hard initial conditions)"
    K = _K()
    K.U0 = float(U0); K.LX = float(LX); K.LY = float(LY); K.H0 = float(H0); K.G = float(G); K.F0 = float(F0); K.OM = float(OMEGA)
    K.T = float(CFG["N_STEPS"] * DT); K.TR = float(RAMP_HOURS * 3600.0); K.TG = float(pc["t_g_h"] * 3600.0); K.n_harm = int(pc["n_harm"])
    # a-priori magnitudes from the forward-solver calibration table (sd of the clean signal ~ 2.8 U0 m, 0.4 U0 m/s, 0.010 U0 m/s)
    K.HS = float(pc["hs_factor"] * U0); K.US = float(pc["us_factor"] * U0); K.VS = float(pc["vs_factor"] * U0)
    K.SV = K.VS * K.OM if pc["v_scale"] == "natural" else K.F0 * K.US      # scale of the v-equation residual
    K.lam_v = float(pc["lam_v"])
    return K

def pinn_features(xp, K, P):
    xh, yh, th = P[..., 0], P[..., 1], P[..., 2]
    t = th * K.T
    f = [xh, yh, th]
    for k in (1, 2):
        f += [xp.cos(k * math.pi * xh), xp.sin(k * math.pi * xh), xp.cos(k * math.pi * yh), xp.sin(k * math.pi * yh)]
    for k in range(1, K.n_harm + 1):
        f += [xp.sin(k * K.OM * t), xp.cos(k * K.OM * t)]
    return xp.stack(f, axis=-1)

def pinn_n_features(K): return 3 + 8 + 2 * K.n_harm

def pinn_mlp(xp, net, z):
    for W, b in net[:-1]: z = xp.tanh(z @ W + b)
    W, b = net[-1]
    return z @ W + b

def pinn_state(xp, K, net, P):
    """P (..., 3) = (x/LX, y/LY, t/T) -> (..., 3) = (eta [m], u [m/s], v [m/s]) with the hard constraints."""
    xh, yh = P[..., 0], P[..., 1]; t = P[..., 2] * K.T
    out = pinn_mlp(xp, net, pinn_features(xp, K, P))
    g = 1.0 - xp.exp(-t / K.TG)
    ramp = xp.where(t < K.TR, 0.5 * (1.0 - xp.cos(math.pi * t / K.TR)), 1.0)
    ubc = K.U0 * ramp * xp.cos(K.OM * t)
    eta = K.HS * g * out[..., 0]
    u = (1.0 - xh) * ubc + K.US * g * 4.0 * xh * (1.0 - xh) * out[..., 1]
    v = K.VS * g * 4.0 * yh * (1.0 - yh) * out[..., 2]
    return xp.stack([eta, u, v], axis=-1)

def cf_point(xp, theta, xh, yh, w):
    """continuous version of cf_zonal: theta = [ln Cf_low, ln Cf_high, b0, s], transition half-width w (normalised x units; classical value 0.5*DX/LX)."""
    arg = (xh - (theta[2] + theta[3] * (yh - 0.5))) / w
    sig = 1.0 / (1.0 + xp.exp(-arg))
    lo, hi = xp.exp(theta[0]), xp.exp(theta[1])
    return lo + (hi - lo) * sig

W_FINAL = 0.5 * DX / LX      # the classical model's transition width

def pinn_residuals(xp, K, S, J, cf):
    """S (...,3) = (eta,u,v); J (...,3,3) with J[...,i,j] = d S_i / d p_j, p = (x/LX, y/LY, t/T). Returns the three normalised residuals."""
    eta, u, v = S[..., 0], S[..., 1], S[..., 2]
    eta_x = J[..., 0, 0] / K.LX; eta_y = J[..., 0, 1] / K.LY; eta_t = J[..., 0, 2] / K.T
    u_x = J[..., 1, 0] / K.LX; u_t = J[..., 1, 2] / K.T
    v_y = J[..., 2, 1] / K.LY; v_t = J[..., 2, 2] / K.T
    fr = cf / K.H0 * xp.sqrt(u * u + v * v + 1e-8)
    Re = (eta_t + K.H0 * (u_x + v_y)) / (K.HS * K.OM)
    Ru = (u_t + K.G * eta_x - K.F0 * v + fr * u) / (K.US * K.OM)
    Rv = (v_t + K.G * eta_y + K.F0 * u + fr * v) / K.SV
    return Re, Ru, Rv

def pinn_jac_fd(K, net, P, h=1e-5):
    """NumPy central finite-difference Jacobian (float64) for tests."""
    cols = []
    for j in range(3):
        dP = np.zeros_like(P); dP[..., j] = h
        cols.append((pinn_state(np, K, net, P + dP) - pinn_state(np, K, net, P - dP)) / (2 * h))
    return np.stack(cols, axis=-1)

def init_net_np(seed, sizes):
    rng = np.random.default_rng(seed); net = []
    for m, n in zip(sizes[:-1], sizes[1:]):
        lim = math.sqrt(6.0 / (m + n)); net.append((rng.uniform(-lim, lim, (m, n)), np.zeros(n)))
    return net

# ---------- data and whitened misfit ----------
def pinn_chi2(xp, Hp, Up, Vp, D, variant):
    """noise-normalised misfit identical to make_chi2 of the twin notebook (exact GLS whitening or white weights); predictions (T, P)."""
    r = (D["Wt"].T @ (Hp - D["h"])) @ D["Ws"]
    c = xp.mean((r * D["inv"]) ** 2)
    if variant == "huv":
        c = (c + xp.mean(((Up - D["u"]) / D["sv"]) ** 2) + xp.mean(((Vp - D["v"]) / D["sv"]) ** 2)) / 3.0
    return c

def build_pinn_data(U0, variant, noise_seed, level, arm):
    """same layout seeds, same noise draws (common random numbers) as the classical rows. Returns dict D (float64 NumPy) and the clean data."""
    P_use = CFG["P_OBS"] if variant == "h" else CFG["P_OBS"] // 3
    set_window(150); set_noise(level)
    ox, oy = _layout_for_seed(P_use, noise_seed); t_idx = REGIMES["rich"]
    if arm == "same":
        Hc, Uc, Vc = run_obs_np(CF_T, U0, CFG["N_STEPS"], ox, oy); clean = dict(h=Hc[t_idx], u=Uc[t_idx], v=Vc[t_idx])
    else:
        Hf, Uf, Vf = gen_fields("standard", U0); clean = dict(h=Hf[t_idx][:, ox, oy], u=Uf[t_idx][:, ox, oy], v=Vf[t_idx][:, ox, oy])
    ctx, obs = make_obs(clean, ox, oy, t_idx, variant, noise_seed, level)
    T, Pn = obs["h"].shape
    th = (np.asarray(t_idx) + 1) * DT / (CFG["N_STEPS"] * DT)          # entry n of every solver/generator = state at time (n+1)*DT
    xh = (np.asarray(ox) + 0.5) / NX; yh = (np.asarray(oy) + 0.5) / NY
    Pd = np.stack([np.tile(xh, T), np.tile(yh, T), np.repeat(th, Pn)], axis=-1)
    Wt = ctx["Wt"] if ctx["Wt"] is not None else np.eye(T); Ws = ctx["Ws"] if ctx["Ws"] is not None else np.eye(Pn)
    inv = ctx["inv"]
    D = dict(Pd=Pd, h=obs["h"], u=obs.get("u", np.zeros((T, Pn))), v=obs.get("v", np.zeros((T, Pn))), Wt=Wt, Ws=Ws, inv=inv, sv=float(ctx["sig_vel"]), T=T, Pn=Pn)
    Dc = dict(D, h=clean["h"], u=clean["u"] if variant == "huv" else np.zeros((T, Pn)), v=clean["v"] if variant == "huv" else np.zeros((T, Pn)))
    # misfit of the TRUE noise realisation = what a perfect solution would give (replaces the classical chi2_ref, which needs a solver at the true parameters)
    chi2_ideal = float(pinn_chi2(np, clean["h"], Dc["u"], Dc["v"], D, variant))
    return D, chi2_ideal, dict(ox=ox, oy=oy, t_idx=t_idx, level=level)

def window_grid(K):
    """all cell centres x all rich-window snapshots: points (T*NX*NY, 3) and the index set, ordering (t, x, y)."""
    t_idx = REGIMES["rich"]; th = (np.asarray(t_idx) + 1) * DT / K.T
    X, Y, Tm = np.meshgrid(XC, YC, th, indexing="ij")
    Pg = np.stack([X, Y, Tm], axis=-1).transpose(2, 0, 1, 3).reshape(-1, 3)       # (T, NX, NY, 3) flattened
    return Pg, t_idx


## G2: residual operator on the generator solution (NumPy)

In [7]:
# ===================== G2: does the continuous residual operator annihilate the generator solution? (NumPy; checks equations, signs and scales) =====================
def gate_residual_on_generator(U0, r=2, n_steps=700, n_lo=300, n_hi=600, verbose=True):
    """generator fields (cell-centred, entry n = state at (n+1) DT), central finite differences on the 24x12 grid; residual of the CONTINUOUS equations in the normalised
    form used by the PINN, with the true friction and with the friction multiplied by 2 (sensitivity)."""
    set_ramp(RAMP_DEFAULT); K = pinn_consts(U0)
    Hh, Uu, Vv = run_generator(TRUTH_FN["standard"], U0, r, n_steps=n_steps)
    X, Y = np.meshgrid(XC, YC, indexing="ij"); cf_true = TRUTH_FN["standard"](X, Y)
    n = np.arange(n_lo, n_hi)
    eta, u, v = Hh[n][:, 1:-1, 1:-1], Uu[n][:, 1:-1, 1:-1], Vv[n][:, 1:-1, 1:-1]
    # derivatives (coarse central differences; time axis: 2 DT central)
    ddx = lambda A: (A[:, 2:, 1:-1] - A[:, :-2, 1:-1]) / (2 * DX)
    ddy = lambda A: (A[:, 1:-1, 2:] - A[:, 1:-1, :-2]) / (2 * DY)
    E, Uf, Vf = Hh[n], Uu[n], Vv[n]
    ddt = lambda A: (A[np.arange(n_lo + 1, n_hi + 1)] - A[np.arange(n_lo - 1, n_hi - 1)]) / (2 * DT)
    Et, Ut, Vt = [ddt(A)[:, 1:-1, 1:-1] for A in (Hh, Uu, Vv)]
    J = np.zeros(eta.shape + (3, 3))
    J[..., 0, 0] = ddx(E) * K.LX; J[..., 0, 1] = ddy(E) * K.LY; J[..., 0, 2] = Et * K.T
    J[..., 1, 0] = ddx(Uf) * K.LX; J[..., 1, 2] = Ut * K.T
    J[..., 2, 1] = ddy(Vf) * K.LY; J[..., 2, 2] = Vt * K.T
    S = np.stack([eta, u, v], -1); out = {}
    for tag, fac in (("true", 1.0), ("x2", 2.0)):
        Re, Ru, Rv = pinn_residuals(np, K, S, J, fac * cf_true[1:-1, 1:-1][None])
        out[tag] = tuple(float(np.sqrt(np.mean(R ** 2))) for R in (Re, Ru, Rv))
    # size of the individual terms (to read the v-equation scale)
    out["v_terms_rms"] = dict(v_t=float(np.sqrt(np.mean((Vt) ** 2))), g_eta_y=float(np.sqrt(np.mean((K.G * ddy(E)) ** 2))), f_u=float(np.sqrt(np.mean((K.F0 * u) ** 2))), SV=K.SV)
    ok = bool(out["true"][0] <= 0.05 and out["true"][1] <= 0.05 and out["x2"][1] >= 1.5 * out["true"][1])
    if verbose:
        print(f"G2 U0={U0}: normalised residual rms (eta, u, v)  true Cf: {out['true'][0]:.4f} {out['true'][1]:.4f} {out['true'][2]:.4f} | Cf x2: {out['x2'][0]:.4f} {out['x2'][1]:.4f} {out['x2'][2]:.4f}")
        print(f"   v-equation terms rms: v_t={out['v_terms_rms']['v_t']:.2e}  g*eta_y={out['v_terms_rms']['g_eta_y']:.2e}  f*u={out['v_terms_rms']['f_u']:.2e}  scale SV={out['v_terms_rms']['SV']:.2e}  ->", "PASS" if ok else "FAIL")
    out["passed"] = ok
    return out


## PINN training (JAX; float32; Adam only)

In [8]:
# ===================== PINN training (JAX, float32, Adam only; no L-BFGS: Radfar 2026) =====================
# STATUS: written without being able to run JAX where it was authored -> everything in this cell is UNTESTED until the gates (stage G) pass on Kaggle.
if HAVE_JAX:
    DTY = jnp.float32
    _tm = jax.tree_util.tree_map
    LO_B = jnp.asarray([b[0] for b in BOUNDS], DTY); HI_B = jnp.asarray([b[1] for b in BOUNDS], DTY)

    def init_net(seed, sizes):
        key = jax.random.PRNGKey(int(seed)); net = []
        for m, n in zip(sizes[:-1], sizes[1:]):
            key, k = jax.random.split(key); lim = math.sqrt(6.0 / (m + n))
            net.append((jax.random.uniform(k, (m, n), DTY, -lim, lim), jnp.zeros((n,), DTY)))
        return net

    def net_sizes(K): return [pinn_n_features(K)] + [int(PINN["width"])] * int(PINN["depth"]) + [3]

    def state_and_jac(K, net, P):
        """(S, J) with J[...,i,j] = d S_i / d p_j via three forward-mode passes (jvp) over all points at once."""
        f = lambda Pp: pinn_state(jnp, K, net, Pp)
        cols = []
        for j in range(3):
            Tj = jnp.zeros_like(P).at[:, j].set(1.0)
            S, dS = jax.jvp(f, (P,), (Tj,)); cols.append(dS)
        return S, jnp.stack(cols, axis=-1)

    def adam(params, grads, m, v, t, lr, b1=0.9, b2=0.999, eps=1e-8):
        m = _tm(lambda a, g: b1 * a + (1.0 - b1) * g, m, grads)
        v = _tm(lambda a, g: b2 * a + (1.0 - b2) * g * g, v, grads)
        c1 = 1.0 - jnp.power(b1, t); c2 = 1.0 - jnp.power(b2, t)
        params = _tm(lambda p, a, b: p - lr * (a / c1) / (jnp.sqrt(b / c2) + eps), params, m, v)
        return params, m, v

    def to_dev(D):
        return {k: jnp.asarray(D[k], DTY) for k in ("Pd", "h", "u", "v", "Wt", "Ws", "inv", "sv")}

    def make_trainer(K, variant, sch, T, Pn):
        """one jitted block of B steps (lax.scan); data, key and schedule are arguments/closure constants -> one compilation per (U0, variant, schedule)."""
        S_ = int(sch["steps"]); B_ = int(sch["block"]); NC = int(PINN["n_col"]); assert S_ % B_ == 0
        def sched(i):
            fi = i.astype(DTY); frac = fi / S_; cosf = 0.5 * (1.0 + jnp.cos(math.pi * frac))
            lr_net = (sch["lr1"] + (sch["lr0"] - sch["lr1"]) * cosf) * jnp.minimum(1.0, (fi + 1.0) / sch["warm"])
            lam = sch["lam0"] * jnp.power(sch["lam1"] / sch["lam0"], jnp.clip(frac / sch["f_lam"], 0.0, 1.0))
            lr_th = jnp.where(frac < sch["f_freeze"], 0.0, sch["lr_theta"] * (0.1 + 0.9 * cosf)) if sch["train_theta"] else 0.0 * frac
            if sch["anneal"]:
                a = jnp.clip((frac - sch["f_freeze"]) / (sch["f_anneal"] - sch["f_freeze"]), 0.0, 1.0)
                w = W_FINAL * jnp.power(sch["w_start"] / W_FINAL, 1.0 - a)
            else:
                w = W_FINAL + 0.0 * frac
            return dict(lr_net=lr_net, lam=lam, lr_th=lr_th, w=w)
        def loss_fn(P, key, sc, D):
            Pc = jax.random.uniform(key, (NC, 3), DTY)
            S, J = state_and_jac(K, P["net"], Pc)
            cf = cf_point(jnp, P["th"], Pc[:, 0], Pc[:, 1], sc["w"])
            Re, Ru, Rv = pinn_residuals(jnp, K, S, J, cf)
            le, lu, lv = jnp.mean(Re ** 2), jnp.mean(Ru ** 2), jnp.mean(Rv ** 2)
            Lp = le + lu + K.lam_v * lv
            if sch["use_data"]:
                Sd = pinn_state(jnp, K, P["net"], D["Pd"])
                Ld = pinn_chi2(jnp, Sd[:, 0].reshape(T, Pn), Sd[:, 1].reshape(T, Pn), Sd[:, 2].reshape(T, Pn), D, variant)
                loss = Ld + sc["lam"] * Lp
            else:
                Ld = jnp.zeros((), DTY); loss = Lp
            return loss, jnp.stack([Ld, le, lu, lv])
        @jax.jit
        def block(carry, idxs, key, D):
            def body(c, i):
                P, opt = c; sc = sched(i); k = jax.random.fold_in(key, i)
                (loss, aux), g = jax.value_and_grad(loss_fn, has_aux=True)(P, k, sc, D)
                t = i.astype(DTY) + 1.0
                net, mn, vn = adam(P["net"], g["net"], opt["m"]["net"], opt["v"]["net"], t, sc["lr_net"])
                th, mt, vt = adam(P["th"], g["th"], opt["m"]["th"], opt["v"]["th"], t, sc["lr_th"])
                th = jnp.clip(th, LO_B, HI_B)
                return (dict(net=net, th=th), dict(m=dict(net=mn, th=mt), v=dict(net=vn, th=vt))), jnp.concatenate([jnp.stack([loss]), aux])
            return lax.scan(body, carry, idxs)
        def run(P0, key, D, tag="", verbose=True):
            opt = dict(m=_tm(jnp.zeros_like, P0), v=_tm(jnp.zeros_like, P0)); carry = (P0, opt); logs = []; t0 = time.time(); nb = S_ // B_
            for b in range(nb):
                carry, lg = block(carry, jnp.arange(b * B_, (b + 1) * B_, dtype=jnp.int32), key, D)
                lgm = np.asarray(lg).mean(0); logs.append([float(x) for x in lgm])
                if verbose and (b % max(1, nb // 6) == 0 or b == nb - 1):
                    print(f"   [{tag}] step {(b + 1) * B_:6d}/{S_}  loss {lgm[0]:.3e}  data {lgm[1]:.3f}  Re {lgm[2]:.2e}  Ru {lgm[3]:.2e}  Rv {lgm[4]:.2e}  [{time.time() - t0:.0f}s]")
            return carry[0], logs
        return run

    _JC = {}
    def eval_states(K, net, P):
        key = ("S", K.U0)
        if key not in _JC: _JC[key] = jax.jit(lambda n, p: pinn_state(jnp, K, n, p))
        return np.asarray(_JC[key](net, jnp.asarray(P, DTY)), dtype=np.float64)
    def eval_resid(K, net, th, Pc, w=None):
        key = ("R", K.U0)
        if key not in _JC:
            def f(n, t, p, ww):
                S, J = state_and_jac(K, n, p); cf = cf_point(jnp, t, p[:, 0], p[:, 1], ww)
                return jnp.stack([jnp.sqrt(jnp.mean(R ** 2)) for R in pinn_residuals(jnp, K, S, J, cf)])
            _JC[key] = jax.jit(f)
        return np.asarray(_JC[key](net, th, jnp.asarray(Pc, DTY), jnp.asarray(W_FINAL if w is None else w, DTY)), dtype=np.float64)

    def sched_dict(steps=None, **kw):
        s = dict(steps=PINN["steps"] if steps is None else steps, block=PINN["block"], warm=PINN["warm"], lr0=PINN["lr0"], lr1=PINN["lr1"],
                 lam0=PINN["lam_p0"], lam1=PINN["lam_p1"], f_lam=PINN["f_lam"], f_freeze=PINN["f_freeze"], f_anneal=PINN["f_anneal"],
                 lr_theta=PINN["lr_theta"], w_start=PINN["w_start"], use_data=True, train_theta=True, anneal=True)
        s.update(kw); return s
    SCH_FORWARD = lambda: sched_dict(lam0=1.0, lam1=1.0, f_lam=1.0, f_freeze=0.0, lr_theta=0.0, use_data=False, train_theta=False, anneal=False)
    def SCH_CONTROL():
        st = max(PINN["block"], int(PINN["steps"] * PINN["ctrl_steps_frac"]) // PINN["block"] * PINN["block"])
        return sched_dict(steps=st, lr0=PINN["ctrl_lr0"], lr1=PINN["ctrl_lr0"] * 0.05, lam0=PINN["lam_p1"], lam1=PINN["lam_p1"], f_lam=1.0,
                          f_freeze=0.0, lr_theta=PINN["lr_theta"] * 0.3, anneal=False)

    def solution_summary(K, P, D, chi2_ideal, variant, Pg, Pc_fixed):
        """everything recorded for one trained network: parameters, misfit, residuals, collapse indicators, validity."""
        th = np.asarray(P["th"], dtype=np.float64); net = P["net"]
        Sd = eval_states(K, net, D["Pd"]); T, Pn = D["h"].shape
        chi2 = float(pinn_chi2(np, Sd[:, 0].reshape(T, Pn), Sd[:, 1].reshape(T, Pn), Sd[:, 2].reshape(T, Pn), D, variant))
        res = eval_resid(K, net, P["th"], Pc_fixed)
        Sg = eval_states(K, net, Pg); rms_u = float(np.sqrt(np.mean(Sg[:, 1] ** 2))); rms_v = float(np.sqrt(np.mean(Sg[:, 2] ** 2)))
        n_obs = int(D["h"].size * (3 if variant == "huv" else 1))
        off = float((th[2] - B0_TRUE) * LX / 1e3); tilt = float((th[3] - S_TRUE) * LX / 1e3)
        finite = bool(np.all(np.isfinite(th)) and np.isfinite(chi2) and np.all(np.isfinite(res)))
        valid = bool(finite and chi2 <= chi2_ideal + 5 * math.sqrt(2.0 / n_obs) and rms_u >= 0.2 * K.U0 and rms_v >= 0.0025 * K.U0)
        return dict(cf_low=float(np.exp(th[0])), cf_high=float(np.exp(th[1])), b0=float(th[2]), s=float(th[3]), chi2=chi2, n_obs=n_obs,
                    bnd_km=float(np.mean(np.abs((th[2] - B0_TRUE) + (th[3] - S_TRUE) * (YC - 0.5))) * LX / 1e3), off_km=off, tilt_km=tilt,
                    res_eta=float(res[0]), res_u=float(res[1]), res_v=float(res[2]), rms_u=rms_u, rms_v=rms_v, finite=finite, valid=valid)

def classify_pinn_seed(sd, tol_cf=0.25, tol_bnd=6.25):
    sols = [s for s in sd["sols"] if s["valid"]]
    if not sols: return "optimiser_failed"
    best = min(s["chi2"] for s in sols); n = sd["n_obs"]
    equiv = [s for s in sols if n * (s["chi2"] - best) <= CFG["DCHI2_CRIT"]]
    ok = all(s["bnd_km"] <= tol_bnd and abs(s["cf_low"] / CF_LOW_TRUE - 1) <= tol_cf and abs(s["cf_high"] / CF_HIGH_TRUE - 1) <= tol_cf for s in equiv)
    return "identified" if ok else "not_identified"

def classify_pinn_row(per_seed):
    labs = [classify_pinn_seed(sd) for sd in per_seed]
    ni, nn, nf = labs.count("identified"), labs.count("not_identified"), labs.count("optimiser_failed"); valid = ni + nn
    if valid < 0.8 * len(per_seed) or valid == 0: cls = "INVALID"
    elif ni >= math.ceil(0.9 * valid): cls = "IDENTIFIED"
    elif ni <= math.floor(0.1 * valid): cls = "NOT_IDENTIFIED"
    else: cls = "MARGINAL"
    return dict(identified=ni, not_identified=nn, optimiser_failed=nf, n_seeds=len(per_seed), cls=cls)


## PINN gates G1, G3, G4 (stage G)

In [9]:
# ===================== PINN gates (stage G): G1 autodiff, G2 residual operator (NumPy, in the cell above), G3 forward test, G4 positive control =====================
PC_FIXED = np.random.default_rng(5).uniform(0.0, 1.0, (16384, 3))        # fixed collocation set for final residual statistics

def _sig_level(level):
    sp = NOISE_LEVELS[level]; return math.hypot(sp["sig_h_white"], sp["sig_h_corr"]), sp["sig_vel"]

if HAVE_JAX:
    def pinn_G1(K):
        """autodiff (forward-mode jvp) Jacobian of the network state vs float64 NumPy central differences of the same function."""
        net = init_net(0, net_sizes(K)); Pt = np.random.default_rng(3).uniform(0.05, 0.95, (256, 3))
        S_j, J_j = jax.jit(lambda n, p: state_and_jac(K, n, p))(net, jnp.asarray(Pt, DTY))
        net64 = [(np.asarray(W, np.float64), np.asarray(b, np.float64)) for W, b in net]
        J_fd = pinn_jac_fd(K, net64, Pt, 1e-5); S_np = pinn_state(np, K, net64, Pt)
        J_j = np.asarray(J_j, np.float64); S_j = np.asarray(S_j, np.float64)
        errJ = max(float(np.max(np.abs(J_j[:, i, j] - J_fd[:, i, j])) / (np.max(np.abs(J_fd[:, i, j])) + 1e-30)) for i in range(3) for j in range(3))
        errS = max(float(np.max(np.abs(S_j[:, i] - S_np[:, i])) / (np.max(np.abs(S_np[:, i])) + 1e-30)) for i in range(3))
        ok = bool(errJ <= 2e-2 and errS <= 1e-3)
        print(f"G1 autodiff vs finite difference: Jacobian max rel err {errJ:.2e} (<= 2e-2), state float32 vs float64 {errS:.2e} (<= 1e-3) ->", "PASS" if ok else "FAIL")
        return dict(err_jac=errJ, err_state=errS, passed=ok)

    def pinn_G3(K, level):
        """forward test: true friction, PDE only (no data), hard BC/IC; error vs the independent generator on the whole window grid, in units of the noise sd of `level`."""
        set_window(150); Pg, t_idx = window_grid(K)
        Hf, Uf, Vf = gen_fields("standard", K.U0)
        truth = np.stack([Hf[t_idx].reshape(-1), Uf[t_idx].reshape(-1), Vf[t_idx].reshape(-1)], -1)
        D0, _, _ = build_pinn_data(K.U0, "h", 0, level, ARM_PINN); T, Pn = D0["h"].shape          # data are not used; only the array shapes
        run = make_trainer(K, "h", SCH_FORWARD(), T, Pn)
        P0 = dict(net=init_net(PINN["net_seed"] + 1, net_sizes(K)), th=jnp.asarray(P_TRUE, DTY))
        P1, logs = run(P0, jax.random.PRNGKey(1), to_dev(D0), tag="G3 forward")
        err = eval_states(K, P1["net"], Pg) - truth; rms = lambda a: float(np.sqrt(np.mean(a ** 2)))
        e_h, e_u, e_v = rms(err[:, 0]), rms(err[:, 1]), rms(err[:, 2]); rel_v = e_v / rms(truth[:, 2])
        sig_h, sig_v = _sig_level(level)
        res = eval_resid(K, P1["net"], P1["th"], PC_FIXED)
        ok = bool(np.isfinite([e_h, e_u, e_v]).all() and e_h <= 0.5 * sig_h and e_u <= 0.5 * sig_v and rel_v <= 0.5)
        in_sigma = {l: dict(h=e_h / _sig_level(l)[0], u=e_u / _sig_level(l)[1]) for l in NOISE_LEVELS}
        print(f"G3 forward test ({level}): rms error  h {e_h*100:.2f} cm ({e_h/sig_h:.2f} sd; need <= 0.5)  u {e_u*100:.2f} cm/s ({e_u/sig_v:.2f} sd; need <= 0.5)  v relative {rel_v:.2f} (need <= 0.5)  residual rms (eta,u,v) = {res.round(5).tolist()}  ->", "PASS" if ok else "FAIL")
        print("   forward error in units of each level's sd (h, u):", {l: (round(v['h'], 2), round(v['u'], 2)) for l, v in in_sigma.items()})
        return dict(rms_h_m=e_h, rms_u_ms=e_u, rms_v_ms=e_v, rel_v=rel_v, residual_rms=res.tolist(), sig_h=sig_h, sig_vel=sig_v, in_sigma=in_sigma, level=level, passed=ok), P1, logs

    def pinn_G4(K, level, P_fwd):
        """positive control: start from the forward solution and the true parameters, let theta and the network adapt to the noisy data of realisation 0; theta must stay identified."""
        D, c2i, _ = build_pinn_data(K.U0, "h", 0, level, ARM_PINN); T, Pn = D["h"].shape
        run = make_trainer(K, "h", SCH_CONTROL(), T, Pn); Pg, _ = window_grid(K)
        P1, _ = run(dict(net=P_fwd["net"], th=jnp.asarray(P_TRUE, DTY)), jax.random.PRNGKey(2), to_dev(D), tag="G4 control")
        sol = solution_summary(K, P1, D, c2i, "h", Pg, PC_FIXED)
        ok = bool(sol["valid"] and sol["bnd_km"] <= 6.25 and abs(sol["cf_low"] / CF_LOW_TRUE - 1) <= 0.25 and abs(sol["cf_high"] / CF_HIGH_TRUE - 1) <= 0.25)
        print(f"G4 positive control ({level}): Cf_low {sol['cf_low']:.2e} (true 1.5e-3)  Cf_high {sol['cf_high']:.2e} (true 4.0e-3)  boundary error {sol['bnd_km']:.2f} km  chi2 {sol['chi2']:.3f} (ideal {c2i:.3f})  valid={sol['valid']}  ->", "PASS" if ok else "FAIL")
        return dict(sol=sol, chi2_ideal=c2i, passed=ok)

def run_pinn_gates():
    G_ = RESULTS["G"].setdefault(PINN_HASH, {}); t0 = time.time()
    set_ramp(RAMP_DEFAULT); set_window(150); set_noise(NOISE_LEVEL)
    U0 = CFG["U0_P"][0]; K = pinn_consts(U0)
    G_["U0"] = U0; G_["G2"] = gate_residual_on_generator(U0)
    if HAVE_JAX:
        G_["G1"] = pinn_G1(K)
        g3, P1, logs = pinn_G3(K, NOISE_LEVEL); G_["G3"] = g3; G_["G3_logs"] = logs
        G_["G4"] = pinn_G4(K, NOISE_LEVEL, P1)
        G_["passed"] = bool(G_["G1"]["passed"] and G_["G2"]["passed"] and G_["G3"]["passed"] and G_["G4"]["passed"])
    else:
        G_["passed"] = False; print("JAX unavailable: G1, G3, G4 not run (NumPy-only G2 ran).")
    RESULTS["G_attempts"].append(dict(hash=PINN_HASH, pinn=dict(PINN), level=NOISE_LEVEL, passed=G_["passed"], G3=G_.get("G3"), mode=MODE))
    save_results()
    print("PINN GATES", "PASSED" if G_["passed"] else "FAILED", f"[{time.time()-t0:.0f}s]  (hyper-parameter hash {PINN_HASH}; every attempt is logged in RESULTS['G_attempts'])")

if STAGE in ("G", "ALL"):
    run_pinn_gates()


G2 U0=1.0: normalised residual rms (eta, u, v)  true Cf: 0.0158 0.0028 0.0116 | Cf x2: 0.0158 0.0484 0.0687
   v-equation terms rms: v_t=1.36e-06  g*eta_y=2.75e-05  f*u=2.78e-05  scale SV=1.41e-06  -> PASS
G1 autodiff vs finite difference: Jacobian max rel err 1.91e-06 (<= 2e-2), state float32 vs float64 8.97e-07 (<= 1e-3) -> PASS
   [G3 forward] step    500/40000  loss 2.071e+03  data 0.000  Re 1.17e+00  Ru 3.35e-01  Rv 2.07e+03  [9s]
   [G3 forward] step   7000/40000  loss 2.156e+00  data 0.000  Re 7.14e-01  Ru 1.49e-01  Rv 1.29e+00  [79s]
   [G3 forward] step  13500/40000  loss 1.923e+00  data 0.000  Re 9.39e-01  Ru 1.15e-01  Rv 8.70e-01  [153s]
   [G3 forward] step  20000/40000  loss 9.655e-01  data 0.000  Re 7.11e-01  Ru 1.12e-01  Rv 1.42e-01  [227s]
   [G3 forward] step  26500/40000  loss 3.496e-01  data 0.000  Re 1.67e-01  Ru 2.02e-02  Rv 1.62e-01  [300s]
   [G3 forward] step  33000/40000  loss 6.310e-02  data 0.000  Re 1.84e-02  Ru 1.43e-02  Rv 3.04e-02  [373s]
   [G3 forward] 

## PINN inversion rows (stages R, S) and classical control rows (stage K)

In [10]:
# ===================== PINN inversion rows (stages R and S) =====================
def pinn_gate_ok():
    if MODE == "quick":
        print("SMOKE TEST (quick mode): the PINN rows run WITHOUT the gate only to exercise the code path; their results are NOT interpretable.")
        return True
    g = RESULTS["G"].get(PINN_HASH, {})
    ok = bool(g.get("passed")) and bool(RESULTS["gate"].get("noise_selftest", False)) and (ARM_PINN == "same" or bool(RESULTS["gate"].get("generator_ok", False)))
    if not ok: print("PINN rows blocked: the gates (stage G: G1-G4 + noise self-test + generator gate) have not been passed under hyper-parameter hash", PINN_HASH)
    return ok

if HAVE_JAX:
    def run_pinn_row(store, tag, U0, variant, level, n_real, n_init, arm=None):
        arm = ARM_PINN if arm is None else arm
        K = pinn_consts(U0); set_ramp(RAMP_DEFAULT); set_window(150)
        rows = store.setdefault(tag, dict(U0=U0, variant=variant, level=level, arm=arm, weighting=WEIGHTING, pinn_hash=PINN_HASH, seeds=[]))["seeds"]
        Pg, _ = window_grid(K); run = None; t0 = time.time()
        for ns in range(len(rows), n_real):
            D, c2i, _ = build_pinn_data(U0, variant, ns, level, arm); Dj = to_dev(D); T, Pn = D["h"].shape
            if run is None: run = make_trainer(K, variant, sched_dict(), T, Pn)
            sols = []
            for k in range(n_init):
                r = np.random.default_rng(100 + k + 17 * ns)             # same start distribution (and seeds) as the classical multi-start
                th0 = np.array([np.log(2.5e-3) + r.normal(0, 0.4), np.log(2.5e-3) + r.normal(0, 0.4), r.uniform(0.3, 0.7), r.uniform(-0.2, 0.2)])
                net = init_net(PINN["net_seed"] + 1000 * ns + k + 1, net_sizes(K))
                P1, _ = run(dict(net=net, th=jnp.asarray(th0, DTY)), jax.random.PRNGKey(10000 + 100 * ns + k), Dj, tag=f"{tag} ns={ns} init={k}", verbose=(ns == 0 and k == 0))
                sol = solution_summary(K, P1, D, c2i, variant, Pg, PC_FIXED); sol["th0"] = th0.tolist(); sols.append(sol)
                print(f"   ns={ns} init={k}: Cf_low {sol['cf_low']:.2e} Cf_high {sol['cf_high']:.2e} bnd {sol['bnd_km']:5.2f} km  chi2 {sol['chi2']:.3f} (ideal {c2i:.3f})  res(e,u,v) {sol['res_eta']:.1e} {sol['res_u']:.1e} {sol['res_v']:.1e}  valid={sol['valid']}")
            rows.append(dict(noise_seed=ns, chi2_ideal=c2i, n_obs=sols[0]["n_obs"], sols=sols, level=level, arm=arm)); save_results()
        c = classify_pinn_row(rows); lo, hi = cp_interval(c["identified"], c["n_seeds"])
        print(f"[PINN row] {tag:44s} identified {c['identified']:2d}/{c['n_seeds']}  CP95=[{lo:.2f},{hi:.2f}]  -> {c['cls']:15s} optimiser_failed={c['optimiser_failed']}  [{time.time()-t0:.0f}s]")
        return c

    def run_pinn_rows(level):
        if not pinn_gate_ok(): return
        for U0 in CFG["U0_P"]:
            for variant in ("h", "huv"):
                run_pinn_row(RESULTS["P"], f"pinn|{ARM_PINN}|{level}|U0={U0}|{variant}|early", U0, variant, level, CFG["N_REAL"], CFG["N_INIT"])
    if STAGE in ("R", "ALL"): run_pinn_rows(NOISE_LEVEL)
    if STAGE == "S": run_pinn_rows("L2")
else:
    print("PINN rows skipped (JAX unavailable).")

# ===================== classical control rows with the independent generator (stage K): same classical code as the twin notebook =====================
if STAGE in ("K", "ALL") and HAVE_JAX and RESULTS["gate"].get("passed"):
    set_ramp(RAMP_DEFAULT)
    for U0 in CFG["U0_P"]:
        for variant in ("h", "huv"):
            run_row(RESULTS["K"], f"classical|indep|{NOISE_LEVEL}|U0={U0}|{variant}|early", U0, variant, CFG["N_SEEDS"], 150, arm="indep", level=NOISE_LEVEL)
    save_results()
elif STAGE in ("K", "ALL"):
    print("classical control rows skipped (JAX unavailable or the solver/generator gate not passed).")


## SUMMARY: tables, pre-registered readings, figures

In [11]:
# ===================== SUMMARY: tables, pre-registered readings, figures =====================
for _p in sorted(glob.glob(os.path.join(OUT_DIR, f"pinn_results_{MODE}_{NOISE_LEVEL}_stage*.json")) + glob.glob(f"/kaggle/input/**/pinn_results_{MODE}_{NOISE_LEVEL}_stage*.json", recursive=True)):
    try:
        _o = json.load(open(_p))
        if _o.get("prereg_hash") != PREREG_HASH or _o.get("mode") != MODE: continue
        for _k in ("P", "K", "G"):
            for _t, _v in _o.get(_k, {}).items():
                if _t not in RESULTS[_k]: RESULTS[_k][_t] = _v; print("filled", _k, _t, "from", os.path.basename(_p))
        if not RESULTS["gate"] and _o.get("gate"): RESULTS["gate"] = _o["gate"]
    except Exception as _e: print("could not read", _p, _e)

print("=" * 110); print("PINN twin experiment | prereg", PREREG_HASH, "| PINN hash", PINN_HASH, "| mode", MODE, "| level", NOISE_LEVEL, "| PINN data arm", ARM_PINN, "| weighting", WEIGHTING, "| JAX", HAVE_JAX); print("=" * 110)
if MODE == "quick": print("*** QUICK MODE = pipeline smoke test: nothing below is a scientific result ***")
print("PINN hyper-parameters:", PINN)
g = RESULTS["G"].get(PINN_HASH)
print("\nGATES (this hyper-parameter hash)")
if g:
    print("   G1 autodiff   :", {k: g["G1"][k] for k in ("err_jac", "err_state", "passed")} if "G1" in g else "not run")
    print("   G2 residual op:", {"true": [round(x, 4) for x in g["G2"]["true"]], "x2": [round(x, 4) for x in g["G2"]["x2"]], "passed": g["G2"]["passed"]} if "G2" in g else "not run")
    if "G3" in g: print("   G3 forward    :", {k: (round(v, 4) if isinstance(v, float) else v) for k, v in g["G3"].items() if k in ("rms_h_m", "rms_u_ms", "rel_v", "sig_h", "sig_vel", "passed")}, "| residual rms (eta,u,v):", [round(x, 5) for x in g["G3"]["residual_rms"]])
    if "G4" in g: print("   G4 control    :", {k: (round(v, 4) if isinstance(v, float) else v) for k, v in g["G4"]["sol"].items() if k in ("cf_low", "cf_high", "bnd_km", "chi2", "valid")}, "passed", g["G4"]["passed"])
    print("   ALL GATES     :", "PASSED" if g.get("passed") else "NOT PASSED  -> no inverse row may be interpreted")
else: print("   no gate result for this hash")
print("   attempts logged:", len(RESULTS["G_attempts"]), "(hash, passed, forward h-error in cm):", [(a["hash"], a["passed"], (round(100 * a["G3"]["rms_h_m"], 2) if a.get("G3") else None)) for a in RESULTS["G_attempts"]])

def _ov(a, b): return not (a[0] > b[1] or b[0] > a[1])
print("\nPINN ROWS (identified / valid; Clopper-Pearson 95%)")
pc = {}
for tag, row in RESULTS["P"].items():
    c = classify_pinn_row(row["seeds"]); lo, hi = cp_interval(c["identified"], c["n_seeds"]); pc[(row["level"], row["U0"], row["variant"])] = c
    nfail = sum(1 for sd in row["seeds"] for s in sd["sols"] if not s["valid"]); ntot = sum(len(sd["sols"]) for sd in row["seeds"])
    print(f"   {tag:44s} identified {c['identified']:2d}/{c['n_seeds']}  CP=[{lo:.2f},{hi:.2f}]  {c['cls']:15s} invalid runs {nfail}/{ntot}")
print("\nCLASSICAL (twin notebook, arm 'same') REFERENCE and CONTROL ROWS (arm 'indep', this notebook)")
for k, v in CLASSICAL_REF.items(): print(f"   reference {k}: {v[0]}/{v[1]}  CP=[{cp_interval(*v)[0]:.2f},{cp_interval(*v)[1]:.2f}]")
kc = {}
for tag, row in RESULTS["K"].items():
    c = classify_row(row["seeds"]); lo, hi = cp_interval(c["identified"], c["n_seeds"]); kc[(row["level"], row["U0"], row["variant"])] = c
    print(f"   control {tag:44s} identified {c['identified']:2d}/{c['n_seeds']}  CP=[{lo:.2f},{hi:.2f}]  {c['cls']}")

print("\nPRE-REGISTERED READINGS")
for (lvl, U0, var), c in sorted(pc.items()):
    ref = CLASSICAL_REF.get((lvl, U0, var))
    if not ref: continue
    cp_, cc_ = cp_interval(c["identified"], c["n_seeds"]), cp_interval(*ref)
    r = "comparable" if _ov(cp_, cc_) else ("PINN worse" if cp_[1] < cc_[0] else "PINN better")
    print(f"   P1 ({lvl}, U0={U0}, {var}): PINN {c['identified']}/{c['n_seeds']} vs classical reference {ref[0]}/{ref[1]} -> {r}")
    c_cls = "identified" if ref[0] >= math.ceil(0.9 * ref[1]) else ("not identified" if ref[0] <= math.floor(0.1 * ref[1]) else "marginal")
    pinn_cls = c["cls"]
    if c_cls == "identified" and pinn_cls in ("NOT_IDENTIFIED", "INVALID"): p3 = "classical identifies, PINN does not -> PINN optimisation problem (hypothesis 2 refuted)"
    elif c_cls == "identified" and pinn_cls == "IDENTIFIED": p3 = "both identify -> structure identifiable at this level"
    elif c_cls != "identified" and pinn_cls in ("NOT_IDENTIFIED", "INVALID"): p3 = "neither identifies -> information limit at this level"
    else: p3 = f"mixed (classical {c_cls}, PINN {pinn_cls}) -> report, no automatic reading"
    print(f"   P3 ({lvl}, U0={U0}, {var}): {p3}")
for (lvl, U0) in sorted({(a, b) for (a, b, _) in pc}):
    if (lvl, U0, "h") in pc and (lvl, U0, "huv") in pc:
        a, b = pc[(lvl, U0, "h")], pc[(lvl, U0, "huv")]
        print(f"   P2 ({lvl}, U0={U0}): velocity helps the PINN:", "SUPPORTED" if cp_interval(b["identified"], b["n_seeds"])[0] > cp_interval(a["identified"], a["n_seeds"])[1] else "NOT SUPPORTED", f"(h {a['identified']}/{a['n_seeds']}, huv {b['identified']}/{b['n_seeds']})")
save_results()
_snap = os.path.join(OUT_DIR, f"pinn_results_{SNAP_TAG}.json")
with open(_snap, "w") as f: json.dump(RESULTS, f, default=_jsonable)
print("\narchive copy of this session:", _snap, "| cumulative file:", OUT_PATH)

try:
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    items = [("PINN " + t, r, "p") for t, r in RESULTS["P"].items()] + [("CLASSICAL(indep) " + t, r, "k") for t, r in RESULTS["K"].items()]
    refs = [(f"classical ref {k}", v) for k, v in CLASSICAL_REF.items()]
    if items or g:
        fig, ax = plt.subplots(1, 2, figsize=(14, 0.45 * (len(items) + len(refs)) + 3.5))
        ys = 0
        for name, r, kind in items:
            c = classify_pinn_row(r["seeds"]) if kind == "p" else classify_row(r["seeds"]); lo, hi = cp_interval(c["identified"], c["n_seeds"]); p = c["identified"] / c["n_seeds"]
            ax[0].errorbar(p, ys, xerr=[[p - lo], [hi - p]], fmt="o", capsize=3, color=("tab:red" if kind == "p" else "tab:blue")); ax[0].text(1.04, ys, name[-48:], fontsize=6, va="center"); ys += 1
        for name, v in refs:
            lo, hi = cp_interval(*v); p = v[0] / v[1]; ax[0].errorbar(p, ys, xerr=[[p - lo], [hi - p]], fmt="s", capsize=3, color="gray"); ax[0].text(1.04, ys, name, fontsize=6, va="center"); ys += 1
        ax[0].set_xlim(-0.03, 1.03); ax[0].axvline(0.9, color="k", ls="--", lw=0.6); ax[0].axvline(0.1, color="k", ls="--", lw=0.6); ax[0].invert_yaxis(); ax[0].set_yticks([])
        ax[0].set_xlabel("fraction of realisations identified (95% CP)   red: PINN, blue: classical (independent data), gray: classical reference (inverse crime)")
        if g and g.get("G3_logs"):
            lg = np.array(g["G3_logs"]); ax[1].semilogy(np.arange(1, len(lg) + 1) * PINN["block"], lg[:, 2], label="Re (mass)"); ax[1].semilogy(np.arange(1, len(lg) + 1) * PINN["block"], lg[:, 3], label="Ru"); ax[1].semilogy(np.arange(1, len(lg) + 1) * PINN["block"], lg[:, 4], label="Rv")
            ax[1].set_xlabel("training step"); ax[1].set_ylabel("mean squared normalised PDE residual"); ax[1].set_title("G3 forward test: residual history"); ax[1].legend(fontsize=8)
        plt.tight_layout(); _fp = os.path.join(OUT_DIR, f"pinn_summary_{SNAP_TAG}.png"); plt.savefig(_fp, dpi=130); print("figure saved:", _fp)
except Exception as e: print("figure skipped:", e)


PINN twin experiment | prereg b907f835f7a4fb04 | PINN hash 4515e1ed182e5d70 | mode standard | level L1 | PINN data arm indep | weighting gls | JAX True
PINN hyper-parameters: {'width': 128, 'depth': 5, 'n_harm': 3, 'steps': 40000, 'block': 500, 'n_col': 8192, 'lr0': 0.001, 'lr1': 2e-05, 'warm': 500, 'lam_p0': 1.0, 'lam_p1': 1000.0, 'f_lam': 0.6, 'f_freeze': 0.25, 'f_anneal': 0.7, 'lr_theta': 0.003, 'w_start': 0.3, 'lam_v': 1.0, 'v_scale': 'natural', 'hs_factor': 3.0, 'us_factor': 1.0, 'vs_factor': 0.01, 't_g_h': 1.0, 'net_seed': 0, 'ctrl_steps_frac': 0.5, 'ctrl_lr0': 0.0002}

GATES (this hyper-parameter hash)
   G1 autodiff   : {'err_jac': 1.9052074959486358e-06, 'err_state': 8.968430228445468e-07, 'passed': True}
   G2 residual op: {'true': [0.0158, 0.0028, 0.0116], 'x2': [0.0158, 0.0484, 0.0687], 'passed': True}
   G3 forward    : {'rms_h_m': 0.7685, 'rms_u_ms': 0.0908, 'rel_v': 0.9821, 'sig_h': 0.02, 'sig_vel': 0.01, 'passed': False} | residual rms (eta,u,v): [0.11886, 0.10732, 0.11